In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import (Circle, Rectangle, FancyBboxPatch, Wedge,
                                FancyArrowPatch, Polygon)

# ---- palette -------------------------------------------------------------
INK    = "#1f2a37"
CHAMB  = "#eef2f6"
XRAY   = "#c0392b"     # X-ray beam (crimson, the single warm accent)
EBEAM  = "#2f4b8f"     # photoelectron path (navy)
ION    = "#3aa07a"     # Ar+ sputter beam (teal)
ANALY  = "#cfd8e0"
LENS   = "#9aa6b2"
DETBOX = "#3aa07a"
TXT    = "#1f2a37"
ITO_C  = "#cdd9ee"     # ITO layer (light navy tint)
SIOX_C = "#e3eaef"     # SiOx (very light)
MO_C   = "#aeb8c2"     # Mo (metal grey)
SI_C   = "#cdbfa6"     # Si substrate (tan, hatched)

FS_LABEL = 14
FS_SMALL = 12
FS_PM    = 16

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 13})

fig, ax = plt.subplots(figsize=(9.0, 7.4))
ax.set_xlim(0, 12.4); ax.set_ylim(0, 10.6); ax.axis("off")
ax.set_aspect("equal")

def arrow(p0, p1, color=INK, lw=2.2, z=6, ls="-", ms=16):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=ms,
                 color=color, lw=lw, ls=ls, zorder=z, shrinkA=0, shrinkB=0))

def wire(pts, lw=2.1, color=INK, z=2, ls="-"):
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, lw=lw, ls=ls, solid_capstyle="round", zorder=z)

# =====================================================================
# UHV chamber
# =====================================================================
CX, CY, CR = 4.35, 4.85, 3.15
ax.add_patch(Circle((CX, CY), CR, facecolor=CHAMB, edgecolor=INK, lw=2.6, zorder=1))
# UHV chamber label (leader to lower-left rim; text sits in clear space below
# the circle so it does not sit on top of the leader line)
xrim, yrim = CX + CR*np.cos(np.deg2rad(212)), CY + CR*np.sin(np.deg2rad(212))
ax.annotate("UHV chamber", xy=(xrim, yrim), xytext=(0.10, 1.35),
            fontsize=FS_LABEL, color=TXT, va="top", ha="left",
            arrowprops=dict(arrowstyle="-", color=INK, lw=1.3,
                            shrinkB=2, shrinkA=4))

# pumps below chamber
ax.add_patch(Rectangle((CX-0.34, CY-CR-0.42), 0.68, 0.45,
                       facecolor="white", edgecolor=INK, lw=1.9, zorder=1))
ax.add_patch(Rectangle((CX-0.92, CY-CR-0.76), 1.84, 0.34,
                       facecolor=LENS, edgecolor=INK, lw=1.9, zorder=1))
ax.add_patch(Rectangle((CX-0.58, CY-CR-1.48), 1.16, 0.72,
                       facecolor="white", edgecolor=INK, lw=1.9, zorder=1))
ax.text(CX, CY-CR-1.66, "pumps", ha="center", va="top",
        fontsize=FS_LABEL, color=TXT)

# =====================================================================
# Sample = the layered stack, in place on the stage INSIDE the chamber
# =====================================================================
sx, shw = 3.70, 0.92            # stack centre x, half-width
sl, sr = sx-shw, sx+shw
layers = [  # (name, y0, y1, facecolor, hatch)
    ("ITO (50 nm)",     4.55, 5.20, ITO_C,  None),
    (r"SiO$_x$ (25 nm)", 4.18, 4.55, SIOX_C, None),
    ("Mo (150 nm)",     3.55, 4.18, MO_C,   None),
    ("Si substrate",    2.95, 3.55, SI_C,   "//"),
]
for name, y0, y1, fc, hatch in layers:
    ax.add_patch(Rectangle((sl, y0), 2*shw, y1-y0, facecolor=fc,
                 edgecolor=INK, lw=1.8, hatch=hatch, zorder=4))
    # short leader to a label that sits just right of the stack, well inside
    # the chamber wall (kept clear of the circle border)
    ax.annotate(name, xy=(sr, (y0+y1)/2), xytext=(sr+0.22, (y0+y1)/2),
                fontsize=FS_SMALL, color=TXT, va="center", ha="left",
                zorder=7,
                arrowprops=dict(arrowstyle="-", color=INK, lw=1.0))
ST_TOP = 5.20
# sample stage / holder beneath the stack
ax.add_patch(Rectangle((sx-0.55, 2.60), 1.10, 0.35,
                       facecolor=LENS, edgecolor=INK, lw=1.5, zorder=3))

# =====================================================================
# X-ray source: mounted OUTSIDE the chamber wall (upper left); the beam
# enters through the wall and strikes the stack surface. The source body sits
# beyond the circle so it does not lie inside the chamber.
# =====================================================================
ang = np.deg2rad(38)
hx, hy = sl+0.32, ST_TOP+0.12                # hit point at the stack surface
bx1, by1 = hx - 2.85*np.cos(ang), hy + 2.85*np.sin(ang)   # source mouth (outside)
# barrel, drawn pointing back away from the chamber
dpx, dpy = 0.30*np.sin(ang), 0.30*np.cos(ang)
ax.add_patch(Polygon([(bx1-dpx, by1-dpy), (bx1+dpx, by1+dpy),
                      (bx1+dpx + 0.60*np.cos(ang), by1+dpy + 0.60*np.sin(ang)),
                      (bx1-dpx + 0.60*np.cos(ang), by1-dpy + 0.60*np.sin(ang))],
             closed=True, facecolor=LENS, edgecolor=INK, lw=1.7, zorder=3))
ax.text(bx1 + 0.60*np.cos(ang) + 0.05, by1 + 0.60*np.sin(ang) + 0.12,
        "X-ray\nsource", ha="center", va="bottom", fontsize=FS_LABEL, color=TXT)
# beam: from source mouth to the surface
arrow((bx1, by1), (hx, hy), color=XRAY, lw=2.3, ls=(0, (5, 2)), z=6, ms=16)
# h-nu label offset above the beam line (clear of the dotted beam & source)
mx, my = (bx1+hx)/2, (by1+hy)/2
ax.text(mx + 0.46, my + 0.20, r"$h\nu$", ha="center", va="center",
        fontsize=FS_LABEL, color=XRAY, zorder=7)

# =====================================================================
# Ar+ sputter ion beam -> stack surface (steep, from the right side so it
# clears the lens column and its label sits in clear space at the rim)
# =====================================================================
arrow((sx+1.35, ST_TOP+0.78), (sx+0.50, ST_TOP+0.03), color=ION, lw=2.3,
      ls=(0, (2, 2)), z=6, ms=16)
ax.text(sx+1.42, ST_TOP+0.82, r"Ar$^{+}$ sputter", ha="left", va="bottom",
        fontsize=FS_SMALL, color=ION, zorder=7)

# small "increasing etch depth" axis just left of the stack
arrow((sl-0.42, ST_TOP-0.05), (sl-0.42, 3.05), color=INK, lw=1.7, z=5, ms=12)
ax.text(sl-0.68, (ST_TOP+3.05)/2, "etch depth", rotation=90, ha="center",
        va="center", fontsize=FS_SMALL, color=TXT)

# =====================================================================
# photoelectron -> lens -> analyser -> multiplier -> spectrum
# =====================================================================
LCX = sx
# photoelectron leaves the stack surface straight up into the lens column
arrow((sx, ST_TOP+0.05), (LCX, ST_TOP+0.78), color=EBEAM, lw=2.1, z=6, ms=15)
ax.text(sx+0.30, ST_TOP+0.45, r"e$^{-}$", ha="left", va="center",
        fontsize=FS_SMALL, color=EBEAM, zorder=7)
# lens plates (clearly above the stack)
for yy in (ST_TOP+0.95, ST_TOP+1.27, ST_TOP+1.59):
    ax.add_patch(Rectangle((LCX-0.40, yy-0.06), 0.80, 0.13,
                           facecolor=LENS, edgecolor=INK, lw=1.4, zorder=5))
ax.annotate("lens system", xy=(LCX+0.40, ST_TOP+1.40), xytext=(LCX+0.75, ST_TOP+1.70),
            fontsize=FS_LABEL, color=TXT, va="center", ha="left",
            arrowprops=dict(arrowstyle="-", color=INK, lw=1.3))

# hemispherical analyser (above the chamber)
HX, HY = LCX, 9.05
R_out, R_in = 1.12, 0.66
ax.add_patch(Wedge((HX, HY), R_out, 0, 180, width=0.34,
                   facecolor=ANALY, edgecolor=INK, lw=2.1, zorder=3))
ax.add_patch(Wedge((HX, HY), R_in, 0, 180, width=0.30,
                   facecolor=ANALY, edgecolor=INK, lw=2.1, zorder=3))
ax.text(HX, HY + R_out + 0.16, "electron energy analyser",
        ha="center", va="bottom", fontsize=FS_LABEL, color=TXT)
ax.text(HX, HY + (R_in+R_out)/2 + 0.02, "$-$", ha="center", va="center",
        fontsize=FS_PM, color=INK, zorder=6)
ax.text(HX, HY + R_in - 0.40, "$+$", ha="center", va="center",
        fontsize=FS_PM, color=INK, zorder=6)
th = np.linspace(np.pi, 0, 60); rr = (R_in+R_out)/2
ax.plot(HX + rr*np.cos(th), HY + rr*np.sin(th), color=EBEAM, lw=1.8,
        ls=(0, (4, 2)), zorder=4)
xen = HX - (R_in+R_out)/2; xex = HX + (R_in+R_out)/2
wire([(LCX, ST_TOP+1.65), (LCX, HY-0.55), (xen, HY-0.55), (xen, HY)],
     color=EBEAM, lw=2.1, z=4)

# electron multiplier (detector) to the upper right -- box sized to the text
DETX, DETY = 9.35, HY
DET_HW, DET_HH = 1.30, 0.62
arrow((xex+0.02, HY), (DETX-DET_HW-0.02, DETY), color=EBEAM, lw=2.1, z=5, ms=15)
ax.add_patch(FancyBboxPatch((DETX-DET_HW, DETY-DET_HH), 2*DET_HW, 2*DET_HH,
             boxstyle="round,pad=0.02,rounding_size=0.10",
             facecolor="white", edgecolor=DETBOX, lw=2.3, zorder=4))
ax.text(DETX, DETY, "electron\nmultiplier", ha="center", va="center",
        fontsize=FS_LABEL, color=TXT, zorder=5)

# XPS spectrum panel below the detector -- larger so the trace + label fit
SPW, SPH = 2.60, 1.65
SPX, SPY = DETX-SPW/2, 6.05
arrow((DETX, DETY-DET_HH), (DETX, SPY+SPH+0.02), color=EBEAM, lw=2.1, z=5, ms=14)
ax.add_patch(FancyBboxPatch((SPX, SPY), SPW, SPH,
             boxstyle="round,pad=0.02,rounding_size=0.08",
             facecolor="white", edgecolor=INK, lw=1.9, zorder=4))
gx = np.linspace(0, 1, 200)
def peak(x0, w, a): return a*np.exp(-((gx-x0)/w)**2)
gy = 0.12 + peak(0.34, 0.05, 0.58) + peak(0.64, 0.07, 0.95)
ax.plot(SPX + 0.26 + gx*(SPW-0.52), SPY + 0.30 + gy*(SPH-0.74),
        color=DETBOX, lw=1.8, zorder=5)
ax.text(SPX+SPW/2, SPY+SPH-0.06, "XPS spectrum", ha="center", va="top",
        fontsize=FS_SMALL, color=TXT, zorder=6)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

INK   = "#2b2b2b"
MO    = "#9aa6b2"
OXIDE = "#eef1f4"
GOLD  = "#e0a93b"
ITO   = "#3aa07a"
ITOr  = "#6b4f2a"
TRAP  = "#3aa07a"
PROT  = "#c0392b"
GREEN = "#3aa07a"
NEU   = "#9aa6b2"

TRAP_PTS = None   # shared trap distribution, set in main()

def _stack(ax, reduced, top_color, top_label):
    """One MIM stack centred in the axes: Mo (grounded) / SiOx / top electrode (-V).

    Bias convention matches the device measurement: a negative bias is applied to
    the top electrode and the Mo bottom electrode is grounded, so the field drives
    the positive mobile species (a proton, with oxygen vacancies as the coupled
    species) UP toward the negatively-biased top electrode."""
    ax.set_xlim(0, 4); ax.set_ylim(0, 3.7); ax.axis("off")
    x0, w = 0.7, 2.6
    y_mo, h_mo = 0.25, 0.45
    y_ox, h_ox = 0.70, 1.7
    y_top, h_top = 2.40, 0.45
    # Mo bottom electrode (grounded)
    ax.add_patch(Rectangle((x0, y_mo), w, h_mo, fc=MO, ec=INK, lw=1.0, zorder=2))
    ax.text(x0+w/2, y_mo+h_mo/2, "Mo", ha="center", va="center", fontsize=8, color="white")
    # ground symbol on Mo
    gx = x0 + w + 0.18
    ax.plot([x0+w, gx], [y_mo+h_mo/2, y_mo+h_mo/2], color=INK, lw=1.0)
    for i, hw in enumerate([0.13, 0.08, 0.04]):
        yy = y_mo+h_mo/2 - 0.10 - 0.06*i
        ax.plot([gx-hw, gx+hw], [yy, yy], color=INK, lw=1.0)
    ax.plot([gx, gx], [y_mo+h_mo/2, y_mo+h_mo/2-0.10], color=INK, lw=1.0)
    # SiOx
    ax.add_patch(Rectangle((x0, y_ox), w, h_ox, fc=OXIDE, ec=INK, lw=1.0, zorder=2))
    ax.text(x0+0.08, y_ox+h_ox-0.12, r"SiO$_x$", ha="left", va="top", fontsize=8, color=INK)
    # top electrode + applied -V
    if reduced:
        ax.add_patch(Rectangle((x0, y_top), w, h_top, fc=ITOr, ec=INK, lw=1.0, zorder=2, hatch="////"))
    else:
        ax.add_patch(Rectangle((x0, y_top), w, h_top, fc=top_color, ec=INK, lw=1.0, zorder=2))
    ax.text(x0+w/2, y_top+h_top/2, top_label, ha="center", va="center", fontsize=8, color="white")
    ax.annotate(r"$-V$ applied", xy=(x0+w/2, y_top+h_top), xytext=(x0+w/2, y_top+h_top+0.55),
                ha="center", fontsize=8.0, color=INK,
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3))
    # distributed traps (same pattern in both)
    xs = x0 + 0.18 + (w-0.36)*TRAP_PTS[:, 0]
    ys = y_ox + 0.20 + (h_ox-0.40)*TRAP_PTS[:, 1]
    ax.scatter(xs, ys, s=15, facecolor="white", edgecolor=TRAP, lw=1.0, zorder=4)
    # mobile positive species drifts UP toward the negative top electrode
    for fx in np.linspace(0.28, 0.72, 4):
        x = x0 + fx*w
        ax.annotate("", xy=(x, y_top-0.04), xytext=(x, y_ox+0.35),
                    arrowprops=dict(arrowstyle="-|>", color=PROT, lw=1.3, alpha=0.9))
        ax.text(x+0.07, y_ox+0.62, r"H$^{+}$", color=PROT, fontsize=6.6, va="center", zorder=5)
    return dict(x0=x0, w=w, y_top=y_top, y_ox=y_ox, h_ox=h_ox)

def panel_a(ax):
    ax.text(-0.06, 1.12, "(a)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.text(0.5, 1.02, "Ti/Au: inert contact", transform=ax.transAxes,
            ha="center", fontsize=9.5, fontweight="bold")
    g = _stack(ax, reduced=False, top_color=GOLD, top_label="Au")
    # accumulation layer (+) just under the inert gold
    ax.add_patch(Rectangle((g["x0"], g["y_top"]-0.15), g["w"], 0.15, fc=PROT, ec="none", alpha=0.6, zorder=3))
    for fx in np.linspace(0.15, 0.85, 6):
        ax.text(g["x0"]+fx*g["w"], g["y_top"]-0.075, "+", color="white",
                ha="center", va="center", fontsize=8, fontweight="bold", zorder=5)
    ax.text(2.0, 0.00, r"H$^{+}$ accumulates at inert Au", ha="center", fontsize=8.0, color=PROT)
    ax.text(2.0, -0.30, "$\\rightarrow$ space charge screens the field", ha="center", fontsize=7.8, color=PROT)

def panel_b(ax):
    ax.text(-0.06, 1.12, "(b)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.text(0.5, 1.02, "ITO: reduced by protons", transform=ax.transAxes,
            ha="center", fontsize=9.5, fontweight="bold")
    g = _stack(ax, reduced=True, top_color=ITO, top_label="ITO")
    for fx in np.linspace(0.2, 0.8, 5):
        ax.text(g["x0"]+fx*g["w"], g["y_top"]-0.01, r"$\times$", color=PROT,
                ha="center", va="center", fontsize=9, fontweight="bold", zorder=5)
    ax.text(2.0, 0.00, r"H$^{+}$ reduces ITO, consumed", ha="center", fontsize=8.0, color=GREEN)
    ax.text(2.0, -0.30, "$\\rightarrow$ no space charge accumulates", ha="center", fontsize=7.8, color=GREEN)

def _transient_axes(ax):
    ax.set_xlim(0, 6); ax.set_ylim(0, 1.08)
    ax.set_xlabel("time (s)", fontsize=8.5); ax.set_ylabel(r"$|I|$", fontsize=9)
    ax.tick_params(labelsize=7.5)

def panel_c(ax):
    """Gold transient: rise then slow tau_d decay -> masks the discharge."""
    ax.text(-0.06, 1.12, "(c)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Au: slow decay dominates", fontsize=9.5, fontweight="bold", pad=4)
    t = np.linspace(0, 6, 400)
    rise = 1 - np.exp(-(t/0.35)**2)
    I = rise*np.exp(-t/9.0); I = I/I.max()
    ax.plot(t, I, color=PROT, lw=2.4)
    _transient_axes(ax)
    # (tau_d arrow and "trap discharge hidden underneath" text removed -- the
    #  caption states the slow proton decay masks the faster trap discharge)

def panel_d(ax):
    """ITO transient: bare dispersive discharge exposed -> tau_leak."""
    ax.text(-0.06, 1.12, "(d)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("ITO: bare discharge exposed", fontsize=9.5, fontweight="bold", pad=4)
    t = np.linspace(0, 6, 400)
    rise = 1 - np.exp(-(t/0.30)**2)
    I = rise*np.exp(-(t/1.3)**0.54); I = I/I.max()
    ax.plot(t, I, color=TRAP, lw=2.4)
    _transient_axes(ax)
    # (tau_leak arrow/annotation removed -- the caption states ITO exposes the
    #  bare discharge tau_leak)

def panel_e(ax):
    """Why dispersive: distribution of rates -> stretched exponential."""
    ax.text(-0.06, 1.12, "(e)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Why dispersive", fontsize=9.5, fontweight="bold", pad=4)
    t = np.linspace(0, 6, 400)
    for tt in [0.4, 0.8, 1.6, 3.2]:
        ax.plot(t, np.exp(-t/tt), color=NEU, lw=0.9, alpha=0.6)
    ax.plot(t, np.exp(-(t/1.3)**0.54), color=TRAP, lw=2.6, label=r"sum: $\beta\!=\!0.54$")
    ax.plot(t, np.exp(-t/1.3), color=PROT, lw=1.5, ls=(0, (4, 2)), label=r"single: $\beta\!=\!1$")
    ax.set_xlim(0, 6); ax.set_ylim(0, 1.02)
    ax.set_xlabel("time (arb.)", fontsize=8.5); ax.set_ylabel("norm. $I$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)
    ax.legend(fontsize=7.4, loc="upper right", framealpha=0.92)
    # ("grey: individual release rates" floating text removed -- caption explains
    #  the spread of release rates summing to a stretched exponential)

def panel_f(ax):
    """One physics, two faces: compressed fill mirrors stretched discharge."""
    ax.text(-0.06, 1.12, "(f)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("One physics, two faces", fontsize=9.5, fontweight="bold", pad=4)
    t = np.linspace(0, 1, 200)
    ax.plot(t, 1-np.exp(-(t/0.18)**2), color=GOLD, lw=2.4, label=r"fill $\beta\!\approx\!2$")
    ax.plot(t, np.exp(-(t/0.30)**0.54), color=TRAP, lw=2.4, label=r"discharge $\beta\!\approx\!0.5$")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.set_xlabel("time (arb.)", fontsize=8.5); ax.set_ylabel("norm. $I$", fontsize=8.5)
    ax.set_xticks([0, 0.5, 1.0]); ax.tick_params(labelsize=7.5)
    # floating "fill/discharge" labels and the italic subtitle removed; the two
    # curves are distinguished by a compact legend and explained in the caption
    ax.legend(fontsize=7.4, loc="center right", framealpha=0.92)

global TRAP_PTS
TRAP_PTS = np.random.RandomState(7).rand(11, 2)
fig, axes = plt.subplots(2, 3, figsize=(11.0, 6.4))
panel_a(axes[0, 0]); panel_b(axes[0, 1]); panel_e(axes[0, 2])
panel_c(axes[1, 0]); panel_d(axes[1, 1]); panel_f(axes[1, 2])
fig.subplots_adjust(left=0.06, right=0.97, top=0.92, bottom=0.12, wspace=0.32, hspace=0.42)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle, Rectangle, FancyBboxPatch

INK   = "#2b2b2b"
CB    = "#c8d0d8"
TRAP  = "#3aa07a"     # volatile tag / trapped electrons
ELEC  = "#3aa07a"
FIL   = "#6b4f2a"     # filament / structural non-volatile weight
CAP   = "#c0392b"     # reward / capture-write
GREEN = "#3aa07a"     # coincidence
ASSUM = "#b07cc6"     # ASSUMED coupling (purple, to flag prediction)
NEU   = "#9aa6b2"

def _trapped_electrons(ax, x0, y0, n, filled):
    """Row of trap sites; 'filled' of n carry an electron (the surviving tag)."""
    xs = np.linspace(x0, x0+2.0, n)
    for i, x in enumerate(xs):
        ax.plot([x-0.12, x+0.12], [y0, y0], color=TRAP, lw=2.0, zorder=3)
        if i < filled:
            ax.add_patch(Circle((x, y0+0.13), 0.10, fc=ELEC, ec=INK, lw=0.7, zorder=5))
    return xs

def _filament(ax, cx, y_bot, y_top, strength):
    """A vertical filament whose width/height encodes the non-volatile weight G."""
    w = 0.10 + 0.28*strength
    ax.add_patch(FancyBboxPatch((cx-w/2, y_bot), w, (y_top-y_bot)*(0.4+0.6*strength),
                 boxstyle="round,pad=0.01", fc=FIL, ec=INK, lw=1.0, zorder=4))

def panel_a(ax):
    ax.text(-0.06, 1.10, "(a)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Two state variables in one cell", fontsize=9.5, fontweight="bold", pad=4)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
    # electrodes
    ax.add_patch(Rectangle((1.0, 8.4), 8.0, 0.5, fc=NEU, ec=INK, lw=1.0)); ax.text(9.1, 8.65, "top", fontsize=7.2, va="center")
    ax.add_patch(Rectangle((1.0, 0.7), 8.0, 0.5, fc=NEU, ec=INK, lw=1.0)); ax.text(9.1, 0.95, "Mo", fontsize=7.2, va="center")
    ax.text(0.9, 4.8, r"SiO$_x$", fontsize=8, ha="right", color=INK)
    # volatile tag: trapped electrons (left)
    _trapped_electrons(ax, 1.7, 5.2, 4, filled=3)
    ax.text(2.5, 6.1, "volatile tag", color=TRAP, fontsize=8.4, ha="center", fontweight="bold")
    ax.text(2.5, 3.9, "trapped electrons\n$e(t)$, decays over $\\tau_{\\mathrm{leak}}$", color=TRAP,
            fontsize=7.4, ha="center")
    # non-volatile weight: filament drawn at the far right, labels to its LEFT (clear of it)
    _filament(ax, 8.5, 1.2, 8.4, strength=0.45)
    ax.text(6.6, 6.1, "non-volatile weight", color=FIL, fontsize=8.4, ha="center", fontweight="bold")
    ax.text(6.6, 3.9, "filament conductance $G$\n(measured ladder)", color=FIL, fontsize=7.4, ha="center")
    ax.annotate("", xy=(8.2, 4.8), xytext=(7.4, 4.8),
                arrowprops=dict(arrowstyle="-", color=FIL, lw=0.8))
    # divider note
    ax.text(5.0, 2.2, "distinct states: the tag alone does not move the weight",
            fontsize=7.0, ha="center", color=NEU, style="italic")

def _cell(ax, e_level, committed, title, ok, note):
    """Shared cell drawing for (b)/(c): tag occupancy e_level, weight grows if committed."""
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
    ax.set_title(title, fontsize=9.5, fontweight="bold", pad=4)
    ax.add_patch(Rectangle((1.0, 8.4), 8.0, 0.5, fc=NEU, ec=INK, lw=1.0))
    ax.add_patch(Rectangle((1.0, 0.7), 8.0, 0.5, fc=NEU, ec=INK, lw=1.0))
    nfill = int(round(4*e_level))
    _trapped_electrons(ax, 1.5, 5.0, 5, filled=nfill)
    ax.text(2.5, 5.9, f"tag $e(t_R)\\!\\approx\\!{e_level:.1f}$", color=TRAP, fontsize=8.0, ha="center")
    # the ASSUMED coupling: trap-assisted switching, drawn as a labelled box that names
    # the physical mechanism (the box IS the trapped-charge gating of the filament write)
    ax.add_patch(FancyBboxPatch((3.3, 6.45), 3.6, 1.35, boxstyle="round,pad=0.04",
                 fc="white", ec=ASSUM, lw=1.6, zorder=5))
    ax.text(5.1, 7.45, "trap-assisted switching", color=ASSUM, fontsize=7.6, ha="center", va="center", fontweight="bold", zorder=7)
    ax.text(5.1, 7.06, r"gate $g(e)$", color=ASSUM, fontsize=7.2, ha="center", va="center", zorder=7)
    ax.text(5.1, 6.70, "(assumed coupling)", color=ASSUM, fontsize=6.6, ha="center", va="center", style="italic", zorder=7)
    # reward pulse arriving from the top electrode, stopping at the box top
    ax.add_patch(FancyArrowPatch((5.1, 9.55), (5.1, 7.86), arrowstyle="-|>",
                 mutation_scale=13, color=CAP, lw=2.0, zorder=6))
    ax.text(5.1, 9.8, "reward pulse", color=CAP, fontsize=7.8, ha="center")
    # tag modulates the gate (left -> gate)
    ax.add_patch(FancyArrowPatch((3.2, 5.3), (3.7, 6.5), arrowstyle="-|>",
                 mutation_scale=9, color=TRAP, lw=1.2, ls=(0,(2,1.3)),
                 connectionstyle="arc3,rad=0.2", zorder=4))
    # gated write: gate -> filament (thickness/alpha scale with how open the gate is)
    a_alpha = 1.0 if ok else 0.22
    ax.add_patch(FancyArrowPatch((6.7, 6.9), (8.3, 4.6), arrowstyle="-|>",
                 mutation_scale=12, color=CAP, lw=1.2+1.8*e_level, alpha=a_alpha,
                 connectionstyle="arc3,rad=-0.25", zorder=5))
    # filament: grows only if committed
    _filament(ax, 8.5, 1.2, 8.4, strength=0.45 + (0.4 if committed else 0.0))
    if committed:
        ax.text(7.0, 3.0, r"$\Delta G$ committed", color=CAP, fontsize=7.8, ha="center", fontweight="bold")
    else:
        ax.text(7.0, 3.0, r"$\Delta G\!\approx\!0$", color=NEU, fontsize=7.8, ha="center")
    ax.text(5.0, 0.12, note, fontsize=7.2, ha="center", color=INK if ok else NEU, style="italic")

def panel_b(ax):
    ax.text(-0.06, 1.10, "(b)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    _cell(ax, e_level=0.8, committed=True, title="Reward within the window", ok=True,
          note="electrons still trapped $\\Rightarrow$ gate open $\\Rightarrow$ structural write")

def panel_c(ax):
    ax.text(-0.06, 1.10, "(c)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    _cell(ax, e_level=0.1, committed=False, title="Reward too late", ok=False,
          note="traps emptied $\\Rightarrow$ gate shut $\\Rightarrow$ no write")

def panel_d(ax):
    ax.text(-0.06, 1.10, "(d)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("The assumed coupling $g(e)$", fontsize=9.5, fontweight="bold", pad=4)
    e = np.linspace(0, 1, 300)
    for p, col, lab in [(1.0, NEU, "$p=1$"), (3.0, ASSUM, "$p=3$ (soft threshold)")]:
        g = e**p/(e**p + 0.25**p)
        ax.plot(e, g, color=col, lw=2.4 if p > 1 else 1.6,
                ls="-" if p > 1 else (0,(4,2)), label=lab)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.set_xlabel(r"surviving tag $e(t_R)$", fontsize=8.5)
    ax.set_ylabel(r"capture fraction $g(e)$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)
    ax.legend(fontsize=7.4, loc="lower right", framealpha=0.92)
    ax.text(0.04, 0.86, r"$\Delta G=\kappa(R\!-\!b)\,g(e)\,\Delta G_0$", transform=ax.transAxes,
            fontsize=8.2, color=CAP)
    ax.text(0.04, 0.70, "single ASSUMED element;\ntarget of the $\\Delta G$-delay\nmeasurement",
            transform=ax.transAxes, fontsize=7.2, color=ASSUM, style="italic")

fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.2))
panel_a(axes[0, 0]); panel_b(axes[0, 1])
panel_c(axes[1, 0]); panel_d(axes[1, 1])
fig.suptitle("Prediction: electron-level capture by a second-order cell [Eq. (10)]",
             fontsize=11, fontweight="bold", y=0.98)
fig.subplots_adjust(left=0.06, right=0.97, top=0.90, bottom=0.09, wspace=0.20, hspace=0.40)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle, Rectangle, FancyBboxPatch

INK   = "#2b2b2b"
CB    = "#cdd6df"     # conduction band shade
TRAP  = "#3aa07a"     # trap level / trapped electrons / trace
ELEC  = "#3aa07a"
CAP   = "#c0392b"     # reward / capture-write
GREEN = "#c75c2e"     # coincidence (spike accent, distinct from teal trace)
NEU   = "#9aa6b2"
PAL   = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]   # tau_leak curves: sequential viridis

def panel_a(ax):
    ax.text(-0.06, 1.10, "(a)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Capture, then emission at a trap", fontsize=9.6, fontweight="bold", pad=4)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")

    # conduction band
    ax.fill_between([0.5, 9.5], 8.2, 10, color=CB, zorder=1)
    ax.plot([0.5, 9.5], [8.2, 8.2], color=INK, lw=1.4, zorder=2)
    ax.text(9.4, 9.1, "conduction band", ha="right", fontsize=8.2, color=INK)

    # A short SEQUENTIAL cascade of trap sites at energetically-distributed depths
    # (faithful to Ch5: carriers hop through a chain v_n1 -> v_n2 -> ... -> v_nk;
    # the sites are spatially and energetically distributed, giving a SPREAD of
    # release times on discharge). Capture fills the chain; emission empties it.
    traps = [(2.2, 6.7), (4.4, 5.7), (6.6, 4.4)]   # distributed depths along the chain
    for j, (x, y) in enumerate(traps):
        ax.plot([x-0.5, x+0.5], [y, y], color=TRAP, lw=2.2, zorder=3)
        ax.add_patch(Circle((x, y+0.17), 0.15, fc=ELEC, ec=INK, lw=0.8, zorder=5))
        ax.text(x, y-0.5, rf"$v_{{n{j+1}}}$", ha="center", fontsize=7.4, color=TRAP)
        # sequential hop to the next site (the cascade rise)
        if j < len(traps)-1:
            xn, yn = traps[j+1]
            ax.add_patch(FancyArrowPatch((x+0.5, y), (xn-0.5, yn), arrowstyle="-|>",
                         mutation_scale=10, color=GREEN, lw=1.4,
                         connectionstyle="arc3,rad=-0.15", zorder=4))
    # capture from band into the chain head (during coincidence)
    ax.add_patch(FancyArrowPatch((2.2, 8.1), (2.2, 6.95), arrowstyle="-|>",
                 mutation_scale=12, color=GREEN, lw=1.7, zorder=4))
    # emission back to band, once the drive stops (dispersive: a spread of rates)
    for (x, y) in traps:
        ax.add_patch(FancyArrowPatch((x+0.30, y+0.30), (x+0.30, 8.1), arrowstyle="-|>",
                     mutation_scale=10, color=CAP, lw=1.2, ls=(0,(2,1.3)), zorder=4))
    # depth bracket: distribution of trap energies
    ax.annotate("", xy=(8.6, 8.2), xytext=(8.6, 4.4),
                arrowprops=dict(arrowstyle="<->", color=INK, lw=1.0))
    ax.text(8.8, 6.3, r"spread of$\ E_T$", fontsize=8.0, color=INK, rotation=90, va="center")
    # labels
    ax.text(2.0, 1.45, "capture (coincidence):", color=GREEN, fontsize=8.0, ha="center")
    ax.text(2.0, 0.95, "fills the sequential chain", color=GREEN, fontsize=7.2, ha="center")
    ax.text(6.8, 1.45, r"emission (drive off): rate $\propto e^{\gamma\sqrt{V}}$", color=CAP, fontsize=8.0, ha="center")
    ax.text(6.8, 0.95, "distributed $E_T$ $\\Rightarrow$ spread of release times", color=CAP, fontsize=7.2, ha="center")

def panel_b(ax):
    ax.text(-0.06, 1.10, "(b)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Trapped population $=$ eligibility trace", fontsize=9.6, fontweight="bold", pad=4)
    dt = 0.01
    t = np.arange(0, 10, dt)
    tau, beta = 2.2, 0.6
    # build during a brief coincidence, then dispersive decay (clamp t-1>=0 so the
    # fractional power never sees a negative base)
    peak = 1 - np.exp(-(1.0/0.18)**2)
    td = np.maximum(t - 1.0, 0.0)
    e = np.where(t < 1.0, 1 - np.exp(-(t/0.18)**2),
                 peak*np.exp(-(td/tau)**beta))
    e = e / e.max()
    ax.fill_between(t, 0, e, color=TRAP, alpha=0.15)
    ax.plot(t, e, color=TRAP, lw=2.4)
    ax.axvspan(0, 1.0, color=GREEN, alpha=0.25)
    ax.text(0.5, 1.06, "coincidence", color=GREEN, fontsize=7.8, ha="center")
    # tau_leak marker at 1/e of peak
    ax.annotate("", xy=(1.0+tau, 0.05), xytext=(1.0, 0.05),
                arrowprops=dict(arrowstyle="<->", color=INK, lw=1.0))
    ax.text(1.0+tau/2, 0.12, r"$\tau_{\mathrm{leak}}=R_{\mathrm{leak}}C$", ha="center", fontsize=8.2, color=INK)
    # (floating "occupancy of trapped electrons = e(t)" annotation removed)
    ax.set_xlim(0, 10); ax.set_ylim(0, 1.12)
    ax.set_xlabel("time after coincidence (s)", fontsize=8.5)
    ax.set_ylabel("trapped population $e(t)$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)

def panel_c(ax):
    ax.text(-0.06, 1.10, "(c)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Reward captures the surviving trace", fontsize=9.6, fontweight="bold", pad=4)
    dt = 0.01
    t = np.arange(0, 10, dt)
    tau, beta = 2.2, 0.6
    peak = 1 - np.exp(-(1.0/0.18)**2)
    td = np.maximum(t - 1.0, 0.0)
    e = np.where(t < 1.0, 1 - np.exp(-(t/0.18)**2),
                 peak*np.exp(-(td/tau)**beta))
    e = e / e.max()
    ax.fill_between(t, 0, e, color=TRAP, alpha=0.15)
    ax.plot(t, e, color=TRAP, lw=2.2)
    # two reward times
    for tR, ok in [(3.0, True), (8.0, False)]:
        eR = e[int(tR/dt)]
        ax.axvline(tR, color=CAP, lw=1.6, ls=(0, (4, 2)), alpha=1.0 if ok else 0.55)
        ax.plot([tR], [eR], "o", color=CAP, ms=6, zorder=5)
        ax.text(tR, 1.04, "reward", color=CAP, fontsize=7.6, ha="center", alpha=1.0 if ok else 0.6)
        # committed weight change bar (proportional to e(t_R))
        ax.add_patch(Rectangle((tR+0.12, 0.0), 0.5, eR*0.8, fc=CAP, ec="none",
                               alpha=0.8 if ok else 0.3, zorder=4))
    # (floating "within window..." and "too late..." annotations removed; the
    #  reward markers and the Delta-w equation below convey the same point)
    ax.text(5.0, -0.30, r"$\Delta w_{ij}=\eta\,(R-b)\,e_{ij}(t_R)$  [Eq.~(3)]",
            transform=ax.transAxes, ha="center", fontsize=8.4, color=CAP)
    ax.set_xlim(0, 10); ax.set_ylim(0, 1.12)
    ax.set_xlabel("action$\\rightarrow$reward delay (s)", fontsize=8.5)
    ax.set_ylabel("eligibility $e(t)$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)

def panel_d(ax):
    ax.text(-0.06, 1.10, "(d)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title(r"$\tau_{\mathrm{leak}}$ sets the credit window", fontsize=9.6, fontweight="bold", pad=4)
    dt = 0.01
    t = np.arange(0, 10, dt)
    peak = 1 - np.exp(-(1.0/0.18)**2)
    for tau, col, lab in [(0.8, PAL[2], r"$\tau_{\mathrm{leak}}$ short (shallow traps)"),
                          (2.2, PAL[1], r"$\tau_{\mathrm{leak}}$ mid"),
                          (5.0, PAL[0], r"$\tau_{\mathrm{leak}}$ long (deep traps)")]:
        td = np.maximum(t - 1.0, 0.0)
        e = np.where(t < 1.0, 1 - np.exp(-(t/0.18)**2),
                     peak*np.exp(-(td/tau)**0.6))
        e = e/e.max()
        ax.plot(t, e, color=col, lw=2.2, label=lab)
    ax.axvspan(0, 1.0, color=GREEN, alpha=0.18)
    ax.set_xlim(0, 10); ax.set_ylim(0, 1.08)
    ax.set_xlabel("time after coincidence (s)", fontsize=8.5)
    ax.set_ylabel("eligibility $e(t)$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)
    ax.legend(fontsize=7.2, loc="upper right", framealpha=0.92)
    # (floating "fast-forgetting <-> integrating" annotation removed; the legend
    #  of short/mid/long tau_leak curves already conveys this)

fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.2))
panel_a(axes[0, 0]); panel_b(axes[0, 1])
panel_c(axes[1, 0]); panel_d(axes[1, 1])
fig.subplots_adjust(left=0.07, right=0.97, top=0.92, bottom=0.10, wspace=0.26, hspace=0.42)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import (FancyBboxPatch, FancyArrowPatch, Circle, Rectangle,
                                Patch, Ellipse, Polygon, Wedge)

INK    = "#2b2b2b"
BIO    = "#b07cc6"
AXON   = "#e7b27a"     # axon terminal
SPINE  = "#b07cc6"     # dendritic spine
TAGB   = "#3aa07a"     # chemical tag (green)
VES    = "#8c5a3c"     # vesicles
DOPA   = "#c0392b"     # dopamine / reward
LTP    = "#b07cc6"     # LTP growth
TRAP   = "#2f4b8f"     # trap / electron
LEAK   = "#c75c2e"     # dispersive leak
REW    = "#c0392b"     # global scalar (R-b)
ELEC   = "#9aa6b2"     # electrode
OX     = "#eef1f4"     # oxide
FIL    = "#6b4f2a"     # filament
PRED   = "#b07cc6"     # predicted
NEU    = "#9aa6b2"

# row y-centres (top -> bottom) and their times -- generously spaced
ROWS = [(13.4, "0 s",  "action"),
        (9.9,  "3 s",  "wait"),
        (6.4,  "5 s",  "reward"),
        (2.6,  "10 s", "capture")]
XL = 3.3     # biology column centre
XR = 10.7    # silicon column centre
XC = 7.0     # timeline x
SY = 0.9     # synapse glyph scale
SC = 1.05    # cell glyph scale

# ----------------------------------------------------------------- synapse glyph
def synapse(ax, cx, cy, grow=0.0, lit=False, tag=None, faded=False):
    """A conventional chemical synapse: a presynaptic axon terminal (bouton) with
    vesicles on top, a synaptic cleft, and a postsynaptic mushroom dendritic spine
    (head on a neck into the dendrite shaft) below, receptors in the head. Signal
    flows top->bottom. Late-LTP enlarges the spine head and adds receptors; the tag
    is a mark in the spine head."""
    s = SY
    # ---- presynaptic axon + bouton (top) ----
    ax.add_patch(Rectangle((cx-0.10*s, cy+1.05*s), 0.20*s, 0.55*s, fc=AXON, ec=INK, lw=1.2, zorder=3))  # axon stalk
    bout_w, bout_h = 0.95*s, 0.62*s
    if lit:
        ax.add_patch(Ellipse((cx, cy+0.62*s), bout_w+0.14*s, bout_h+0.14*s, fc="none", ec="#e0a93b", lw=2.6, zorder=2))
    ax.add_patch(Ellipse((cx, cy+0.62*s), bout_w, bout_h, fc=AXON, ec=INK, lw=1.6, zorder=4))            # bouton
    # vesicles clustered toward the active zone (bottom of the bouton)
    rng = np.random.RandomState(2)
    for i in range(5):
        vx = cx-0.30*s + 0.60*s*rng.rand(); vy = cy+0.45*s + 0.34*s*rng.rand()
        ax.add_patch(Circle((vx, vy), 0.075*s, fc="white", ec=VES, lw=1.0, zorder=5))
    # ---- synaptic cleft ----
    ax.add_patch(Rectangle((cx-0.50*s, cy+0.20*s), 1.0*s, 0.06*s, fc="none", ec="none", zorder=3))
    if lit:  # neurotransmitter released into the cleft
        for k in range(4):
            ax.add_patch(Circle((cx-0.28*s+0.56*s*rng.rand(), cy+0.20*s+0.06*s*rng.rand()),
                         0.05*s, fc=TAGB, ec="none", alpha=0.85, zorder=6))
    # ---- postsynaptic mushroom spine (head + neck into dendrite) ----
    head_r = (0.42 + 0.26*grow)*s
    head_y = cy - 0.10*s - head_r
    ax.add_patch(Rectangle((cx-0.22*s, head_y-0.55*s), 0.44*s, 0.5*s, fc=SPINE, ec=INK, lw=1.2, zorder=2))  # dendrite shaft
    ax.add_patch(Rectangle((cx-0.09*s, head_y-0.10*s), 0.18*s, 0.32*s,
                 fc=(LTP if grow > 0 else SPINE), ec=INK, lw=1.1, zorder=3))                               # neck
    ax.add_patch(Circle((cx, head_y), head_r, fc=(LTP if grow > 0 else SPINE), ec=INK, lw=1.6,
                 alpha=0.95 if grow > 0 else 1.0, zorder=4))                                               # head
    # receptors on the head facing the cleft (top arc; more when grown)
    nrec = 3 + int(round(3*grow))
    for a in np.linspace(-0.7, 0.7, nrec):
        rx = cx + head_r*np.sin(a); ry = head_y + head_r*np.cos(a)
        ax.add_patch(Rectangle((rx-0.045*s, ry-0.05*s), 0.09*s, 0.12*s, fc=INK, ec="none", zorder=5))
    # the chemical tag inside the spine head
    if tag is not None:
        tc = "#a9d9c5" if faded else TAGB
        ax.add_patch(Circle((cx, head_y), 0.15*s, fc=tc, ec=INK, lw=0.9, zorder=6,
                     alpha=0.5 if faded else 1.0))

# ----------------------------------------------------------------- SiOx cell glyph
def cell(ax, cx, cy, fil=0.0, trapped=0):
    s = SC
    ax.add_patch(Rectangle((cx-0.95*s, cy+0.50*s), 1.9*s, 0.20*s, fc=ELEC, ec=INK, lw=0.9, zorder=4))   # top electrode
    ax.add_patch(Rectangle((cx-0.95*s, cy-0.70*s), 1.9*s, 0.20*s, fc=ELEC, ec=INK, lw=0.9, zorder=4))   # Mo bottom
    ax.add_patch(Rectangle((cx-0.95*s, cy-0.50*s), 1.9*s, 1.00*s, fc=OX, ec=INK, lw=0.9, zorder=3))     # SiOx
    ax.text(cx-0.80*s, cy+0.40*s, r"SiO$_x$", fontsize=7.0, color=INK, va="top", zorder=6)
    # filament (thickness = non-volatile weight)
    fw = (0.10 + 0.42*fil)*s
    if fil > 0:
        ax.add_patch(Rectangle((cx+0.55*s-fw/2, cy-0.50*s), fw, 1.00*s, fc=FIL, ec=INK, lw=0.7, zorder=5))
    # trap sites (filled = trapped electrons), left of the filament
    rng = np.random.RandomState(7)
    for i in range(6):
        x = cx-0.62*s + 0.55*s*rng.rand(); y = cy-0.34*s + 0.68*s*rng.rand()
        filled = i < trapped
        ax.add_patch(Circle((x, y), 0.075*s, fc=(TRAP if filled else "white"), ec=TRAP, lw=1.0, zorder=6))

# ----------------------------------------------------------------- main
fig, ax = plt.subplots(figsize=(11.0, 13.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 15.8); ax.axis("off")

# column headers (in the top margin, clear of the first row band which tops at 14.85)
ax.text(XL, 15.5, "BIOLOGY", ha="center", fontsize=15, fontweight="bold", color=BIO)
ax.text(XL, 15.12, "synaptic tagging and capture", ha="center", fontsize=9.5, color=BIO, style="italic")
ax.text(XR, 15.5, "SILICON", ha="center", fontsize=15, fontweight="bold", color=TRAP)
ax.text(XR, 15.12, "subthreshold SiO$_x$ memristor", ha="center", fontsize=9.5, color=TRAP, style="italic")

# segmented timeline: a separate arrow into each node, with the event name
# sitting in the GAP just above the node (no overlap with the arrow or label)
node_r = 0.56
seg_tops = [14.85] + [ROWS[i][0]-node_r for i in range(len(ROWS)-1)]
for (y, t, lab), top in zip(ROWS, seg_tops):
    # faint full-width row band
    ax.add_patch(Rectangle((0.4, y-1.72), 13.2, 3.36, fc="#f4f5f7", ec="none", zorder=0))
    # event-name label sits in the gap above the node, beside the segment
    ax.text(XC, (top+y+node_r)/2, lab, ha="center", va="center", fontsize=11,
            color=INK, fontweight="bold", style="italic", zorder=7,
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none"))
    # arrow for this segment, stopping at the node top
    ax.add_patch(FancyArrowPatch((XC, top), (XC, y+node_r), arrowstyle="-|>",
                 mutation_scale=20, color=INK, lw=2.2, zorder=2))
    # the node circle with the timestamp
    ax.add_patch(Circle((XC, y), node_r, fc="white", ec=INK, lw=2.0, zorder=6))
    ax.text(XC, y, t, ha="center", va="center", fontsize=11.5, fontweight="bold", color=INK, zorder=7)
# tail arrow below the last node
ax.add_patch(FancyArrowPatch((XC, ROWS[-1][0]-node_r), (XC, 0.95), arrowstyle="-|>",
             mutation_scale=20, color=INK, lw=2.2, zorder=2))
ax.text(XC, 0.6, "time", ha="center", fontsize=10.5, color=INK, fontweight="bold")

# ---------------- BIOLOGY ----------------
y = ROWS[0][0]; synapse(ax, XL, y, grow=0.0, lit=True, tag=TAGB)
ax.text(XL, y-1.18, "terminal fires; sets a transient tag", ha="center", fontsize=10.0, color=TAGB)
y = ROWS[1][0]; synapse(ax, XL, y, grow=0.0, tag=TAGB, faded=True)
ax.text(XL, y-1.18, "the chemical tag fades", ha="center", fontsize=10.0, color=TAGB)
y = ROWS[2][0]; synapse(ax, XL, y, grow=0.0, tag=TAGB)
for dx, dy in [(-1.7, 1.0), (-1.1, 1.25), (1.3, 1.05), (1.9, 0.65), (0.5, 1.35), (-0.3, 1.1)]:
    ax.add_patch(Circle((XL+dx, y+dy), 0.12, fc=DOPA, ec="none", alpha=0.85, zorder=5))
ax.add_patch(FancyArrowPatch((XL+0.9, y+1.0), (XL+0.1, y+0.2), arrowstyle="-|>",
             mutation_scale=12, color=DOPA, lw=1.5, zorder=6))
ax.text(XL, y-1.18, "dopamine binds only the surviving tag", ha="center", fontsize=10.0, color=DOPA)
y = ROWS[3][0]; synapse(ax, XL, y, grow=1.0, tag=None)
ax.text(XL, y-1.58, "late-LTP: the spine enlarges", ha="center", fontsize=10.0, color=LTP)

# ---------------- SILICON ----------------
y = ROWS[0][0]; cell(ax, XR, y, fil=0.28, trapped=6)
ax.add_patch(FancyArrowPatch((XR-1.8, y), (XR-1.0, y), arrowstyle="-|>",
             mutation_scale=13, color=TRAP, lw=1.7, zorder=6))
ax.text(XR, y-1.30, "LIF fires; electrons into deep traps", ha="center", fontsize=10.0, color=TRAP)
y = ROWS[1][0]; cell(ax, XR, y, fil=0.28, trapped=2)
for dx in (-0.45, -0.20, 0.05):
    ax.add_patch(FancyArrowPatch((XR+dx, y+0.40), (XR+dx, y+0.82), arrowstyle="-|>",
                 mutation_scale=9, color=LEAK, lw=1.2, ls=(0,(2,1)), zorder=6))
# dispersive-leak inset, tucked into the right margin beside this cell (not floating)
ix, iy, iw, ih = XR+1.25, y-0.45, 1.05, 0.95
tt = np.linspace(0, 1, 60); kww = np.exp(-(tt/0.35)**0.54)
ax.plot(ix+tt*iw, iy+kww*ih, color=LEAK, lw=2.0, zorder=6)
ax.plot([ix, ix, ix+iw], [iy+ih, iy, iy], color=NEU, lw=0.8, zorder=5)   # tiny axes
ax.text(ix+iw*0.55, iy+ih*0.78, r"$\beta\!<\!1$", fontsize=8.6, color=LEAK, ha="left")
ax.text(ix+iw*0.5, iy-0.22, "leak", fontsize=7.2, color=LEAK, ha="center", style="italic")
ax.text(XR, y-1.30, "electrons detrap: dispersive leak", ha="center", fontsize=10.0, color=LEAK)
y = ROWS[2][0]; cell(ax, XR, y, fil=0.28, trapped=2)
# the (R-b) pulse is gated by the surviving tag: it must meet the trapped
# electrons (left of the cell), not act on the filament directly -- the
# filament change is the consequence, shown in the next (capture) row
ax.add_patch(FancyArrowPatch((XR+1.8, y+0.95), (XR-0.25, y+0.18), arrowstyle="-|>",
             mutation_scale=14, color=REW, lw=2.3, zorder=7))
ax.text(XR+1.95, y+1.05, r"$(R\!-\!b)$", ha="center", fontsize=11, color=REW, fontweight="bold")
ax.text(XR, y-1.30, "global scalar meets trapped electrons", ha="center", fontsize=10.0, color=REW)
y = ROWS[3][0]; cell(ax, XR, y, fil=1.0, trapped=1)
ax.text(XR, y-1.30, "non-volatile weight committed", ha="center", fontsize=10.0, color=FIL)
ax.text(XR, y-1.72, r"three-factor write: $\Delta w=\eta(R-b)\,e(t_R)$", ha="center", fontsize=8.8, color=FIL, style="italic")

fig.suptitle("From a synaptic tag to a silicon tag: a biological-to-device translation map",
             fontsize=13.5, fontweight="bold", y=0.975)
fig.subplots_adjust(left=0.01, right=0.99, top=0.93, bottom=0.015)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import (Circle, FancyBboxPatch, FancyArrowPatch,
                                Rectangle)

INK = "#2b2b2b"; CUE = "#e0a93b"; DIST = "#9aa6b2"; REW = "#c0392b"
TRACE = "#3aa07a"; DEV = "#eaf0fb"; OUT = "#cfd8e0"

plt.rcParams.update({"font.family": "DejaVu Sans"})
fig = plt.figure(figsize=(10.0, 4.4))
gsL = fig.add_axes([0.015, 0.04, 0.52, 0.92]); gsL.axis("off")
gsR = fig.add_axes([0.60, 0.04, 0.39, 0.92]); gsR.axis("off")
gsL.set_xlim(0, 10.6); gsL.set_ylim(0, 10)
gsR.set_xlim(0, 10); gsR.set_ylim(0, 10)

def arrow(ax, p0, p1, color=INK, lw=1.8, ms=12, ls="-", z=5, rad=0.0):
    cs = f"arc3,rad={rad}" if rad else None
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=ms,
                 color=color, lw=lw, ls=ls, zorder=z, shrinkA=0, shrinkB=0,
                 connectionstyle=cs))

# ============================ PANEL A ============================
gsL.text(-0.6, 9.7, "(a)", fontsize=15, fontweight="bold", va="top")

# --- network sits in the upper band (y ~ 4.5-9) ---
ins_y = [8.4, 7.3, 6.2, 5.1]
labels = ["cue", "distractor", "distractor", r"$\vdots$"]
incol = [CUE, DIST, DIST, DIST]
ix = 1.5
out = (6.4, 6.75)
synx = (ix + out[0]) / 2          # synapse-box column
bus_x = 8.3                        # reward bus column (right of everything)

for y, lab, c in zip(ins_y, labels, incol):
    if lab == r"$\vdots$":
        gsL.text(ix, y, r"$\vdots$", fontsize=22.4, ha="center", va="center", color=INK)
        continue
    gsL.add_patch(Circle((ix, y), 0.40, facecolor="white", edgecolor=c, lw=2.2, zorder=4))
    gsL.text(ix - 0.70, y, lab, fontsize=14, ha="right", va="center", color=c)
    # synapse box on the wire to the output
    gsL.add_patch(FancyBboxPatch((synx-0.34, y-0.21), 0.68, 0.42,
                  boxstyle="round,pad=0.02,rounding_size=0.06",
                  facecolor=DEV, edgecolor=INK, lw=1.2, zorder=5))
    gsL.plot([ix+0.40, synx-0.34], [y, y], color=c, lw=1.5, zorder=3)
    arrow(gsL, (synx+0.34, y), (out[0]-0.55, out[1]), color=c, lw=1.5, ms=10, z=3)
gsL.text(synx, ins_y[0]+0.62, "device synapses", fontsize=12.6,
         ha="center", color=INK, style="italic")

# LIF output neuron
gsL.add_patch(Circle(out, 0.55, facecolor=OUT, edgecolor=INK, lw=2.4, zorder=4))
gsL.text(out[0], out[1], "LIF", fontsize=14, ha="center", va="center", color=INK, fontweight="bold")
gsL.text(out[0], out[1]-0.85, "output", fontsize=13.3, ha="center", va="top", color=INK)

# --- reward delivered on a single clean vertical bus (no crossing lines) ---
# bus runs down the right side; a short horizontal stub taps each synapse box.
bus_top = ins_y[0] + 0.05
bus_bot = 3.7
gsL.plot([bus_x, bus_x], [bus_bot, bus_top], color=REW, lw=2.0, zorder=2)
for y in ins_y[:3]:
    arrow(gsL, (bus_x, y), (synx+0.36, y), color=REW, lw=1.2, ms=9,
          ls=(0, (3, 2)), z=2)
    gsL.add_patch(Circle((bus_x, y), 0.06, facecolor=REW, edgecolor=REW, zorder=3))
# reward source box feeding the bus from below (centred on the bus, fully inside axes)
rb_w, rb_h = 2.8, 1.0
rb_cx, rb_cy = bus_x, 3.15
gsL.add_patch(FancyBboxPatch((rb_cx-rb_w/2, rb_cy-rb_h/2), rb_w, rb_h,
              boxstyle="round,pad=0.03,rounding_size=0.1",
              facecolor="white", edgecolor=REW, lw=2.0, zorder=5))
gsL.text(rb_cx, rb_cy+0.18, r"reward $R(t)$", fontsize=14, ha="center", va="center", color=REW, zorder=6)
gsL.text(rb_cx, rb_cy-0.22, "global third factor", fontsize=11.2, ha="center", va="center", color=REW, style="italic", zorder=6)
gsL.plot([bus_x, bus_x], [rb_cy+rb_h/2, bus_bot], color=REW, lw=2.0, zorder=2)
gsL.text(bus_x+0.18, (bus_bot+bus_top)/2, "broadcast", fontsize=10.5, ha="left",
         va="center", color=REW, rotation=90, style="italic")

# --- timeline in its own band at the bottom (y ~ 0.4-2.0), well separated ---
ty = 1.1
gsL.annotate("", xy=(9.8, ty), xytext=(0.4, ty),
             arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3))
gsL.text(9.7, ty-0.45, "time", fontsize=11.9, ha="right", color=INK)
# cue epoch block (below the axis), label below it
gsL.add_patch(Rectangle((1.0, ty-0.30), 1.8, 0.30, facecolor=CUE, alpha=0.45, edgecolor="none"))
gsL.text(1.9, ty-0.55, "cue epoch", fontsize=11.9, ha="center", va="top", color=CUE)
# delay span (above the axis), label above it
gsL.annotate("", xy=(7.0, ty+0.30), xytext=(2.8, ty+0.30),
             arrowprops=dict(arrowstyle="<->", color=INK, lw=1.1))
gsL.text(4.9, ty+0.42, r"action$\to$reward delay $D$", fontsize=11.9, ha="center", va="bottom", color=INK)
# reward marker (on the axis), label above the tick, clear of the delay span
gsL.plot([7.0, 7.0], [ty-0.18, ty+0.18], color=REW, lw=2.4)
gsL.text(7.0, ty-0.30, "reward", fontsize=11.9, ha="center", va="top", color=REW)

# ============================ PANEL B ============================
gsR.text(-0.6, 9.7, "(b)", fontsize=15, fontweight="bold", va="top")
# 1. coincidence
gsR.add_patch(FancyBboxPatch((0.5, 8.0), 9.0, 1.05, boxstyle="round,pad=0.03,rounding_size=0.1",
              facecolor="white", edgecolor=INK, lw=1.6))
gsR.text(5.0, 8.72, "pre--post coincidence (signed, leak-dominant)", fontsize=10.8, ha="center", color=INK)
gsR.text(2.6, 8.25, r"causal: $+1$", fontsize=11.9, ha="center", color=CUE)
gsR.text(7.2, 8.25, r"acausal: $-\lambda$", fontsize=11.9, ha="center", color=REW)
arrow(gsR, (5.0, 8.0), (5.0, 7.35), lw=1.6)
# 2. device gate box (widened to match the coincidence box so the subtitle fits)
gsR.add_patch(FancyBboxPatch((0.5, 5.1), 9.0, 2.1, boxstyle="round,pad=0.03,rounding_size=0.1",
              facecolor=DEV, edgecolor=INK, lw=1.8))
gsR.text(5.0, 6.85, "device transient gate", fontsize=13.3, ha="center", color=INK, fontweight="bold")
gsR.text(5.0, 6.40, r"trap cascade ($k\!\approx\!3$, rise) $+$ $R_{\mathrm{leak}}$ (relax)", fontsize=10.8, ha="center", color=INK)
# mini trace inside
tx = np.linspace(0, 1, 120)
e = (1 - np.exp(-(tx/0.18)**2)) * np.exp(-tx/0.45); e = e/e.max()
gsR.plot(2.2 + tx*5.0, 5.35 + e*0.75, color=TRACE, lw=1.8)
gsR.text(7.6, 5.7, r"$e(t)$", fontsize=12.6, color=TRACE, ha="left")
arrow(gsR, (5.0, 5.1), (5.0, 4.45), lw=1.6)
# 3. eligibility trace label
gsR.text(5.0, 4.12, r"eligibility trace $e_{ij}(t)$,  retention $\tau_{\mathrm{leak}}\!=\!R_{\mathrm{leak}}C$",
         fontsize=12.3, ha="center", color=TRACE)
arrow(gsR, (5.0, 3.75), (5.0, 3.05), lw=1.6)
# 4. update box
gsR.add_patch(FancyBboxPatch((1.0, 1.6), 8.0, 1.35, boxstyle="round,pad=0.03,rounding_size=0.1",
              facecolor="white", edgecolor=REW, lw=2.0))
gsR.text(5.0, 2.50, "three-factor update at reward", fontsize=13.3, ha="center", color=REW, fontweight="bold")
gsR.text(5.0, 1.92, r"$\Delta w_{ij} = \eta\,(R-b)\,e_{ij}(t_R)$", fontsize=16.8, ha="center", color=INK)
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import (Circle, FancyBboxPatch, FancyArrowPatch,
                                Rectangle)

INK = "#2b2b2b"; STATE = "#e0a93b"; ACT = "#2f4b8f"; REW = "#c0392b"
TRACE = "#3aa07a"; DEV = "#eaf0fb"; OUT = "#cfd8e0"; GRID = "#9aa6b2"

fig = plt.figure(figsize=(9.8, 4.3))
A = fig.add_axes([0.015, 0.04, 0.575, 0.92]); A.axis("off")
B = fig.add_axes([0.625, 0.04, 0.365, 0.92]); B.axis("off")
A.set_xlim(0, 11); A.set_ylim(0, 9.6)
B.set_xlim(0, 10); B.set_ylim(0, 9)

def arrow(ax, p0, p1, color=INK, lw=1.7, ms=12, ls="-", z=5):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=ms,
                 color=color, lw=lw, ls=ls, zorder=z, shrinkA=0, shrinkB=0))

# ============================ PANEL A: crossbar ============================
A.text(-0.2, 8.8, "(a)", fontsize=15, fontweight="bold", va="top")

rows_y = [7.1, 6.0]                       # two state wordlines
cols_x = [4.2, 5.7]                       # two action bitlines
row_x0, row_x1 = 1.7, 6.6
col_y0, col_y1 = 4.6, 7.6

# state input wordlines (rows)
state_lab = [r"$s_1$", r"$s_2$"]
for y, lab in zip(rows_y, state_lab):
    A.plot([row_x0, row_x1], [y, y], color=STATE, lw=2.0, zorder=2)
    A.add_patch(Circle((row_x0 - 0.45, y), 0.34, facecolor="white",
                       edgecolor=STATE, lw=2.0, zorder=4))
    A.text(row_x0 - 0.45, y, lab, fontsize=9.5, ha="center", va="center", color=STATE)
A.text(row_x0 - 0.95, (rows_y[0] + rows_y[1]) / 2, "state\ninputs", fontsize=8.4,
       ha="right", va="center", color=STATE, style="italic")

# action bitlines (cols) + compound-cell synapses at crosspoints
for x in cols_x:
    A.plot([x, x], [col_y0, col_y1], color=ACT, lw=2.0, zorder=2)

def compound_cell(ax, cx, cy):
    """One synapse drawn as its true compound cell: a non-volatile WEIGHT device (left,
    dark) + a subthreshold TRACE device (right, light) + a small active-gating element
    (the access transistor / selector) marked beneath. Compact so the 2x2 array stays
    legible while honestly showing two devices per synapse rather than one glyph."""
    # weight device (left square, filled)
    ax.add_patch(FancyBboxPatch((cx - 0.30, cy - 0.15), 0.27, 0.30,
                 boxstyle="round,pad=0.01,rounding_size=0.04",
                 facecolor=DEV, edgecolor=INK, lw=1.0, zorder=5))
    # trace device (right square, light fill, trace colour edge)
    ax.add_patch(FancyBboxPatch((cx + 0.03, cy - 0.15), 0.27, 0.30,
                 boxstyle="round,pad=0.01,rounding_size=0.04",
                 facecolor="white", edgecolor=TRACE, lw=1.3, zorder=5))
    # active-gating element: a small filled circle (transistor/selector) under the pair,
    # tapping the third-factor line
    ax.add_patch(Circle((cx, cy - 0.32), 0.07, facecolor=REW, edgecolor=INK,
                        lw=0.8, zorder=6))

for y in rows_y:
    for x in cols_x:
        compound_cell(A, x, y)
# compact inline legend to the RIGHT of the array (clear of the left-margin labels),
# stacked above the I=G^T V annotation
leg_x = cols_x[1] + 1.15
leg_y = col_y1 + 0.55
A.add_patch(FancyBboxPatch((leg_x, leg_y - 0.10), 0.20, 0.22, boxstyle="round,pad=0.01",
            facecolor=DEV, edgecolor=INK, lw=0.8, zorder=5))
A.text(leg_x + 0.30, leg_y, "weight device", fontsize=6.8, ha="left", va="center", color=INK)
A.add_patch(FancyBboxPatch((leg_x, leg_y - 0.50), 0.20, 0.22, boxstyle="round,pad=0.01",
            facecolor="white", edgecolor=TRACE, lw=1.0, zorder=5))
A.text(leg_x + 0.30, leg_y - 0.40, "trace device", fontsize=6.8, ha="left", va="center", color=TRACE)
A.add_patch(Circle((leg_x + 0.10, leg_y - 0.80), 0.07, facecolor=REW, edgecolor=INK, lw=0.7, zorder=6))
A.text(leg_x + 0.30, leg_y - 0.80, "active gating", fontsize=6.8, ha="left", va="center", color=REW)
A.text(leg_x, leg_y + 0.40, "each synapse:", fontsize=7.0, ha="left", va="center",
       color=INK, style="italic")

# bitline current accumulation label
A.text(cols_x[1] + 1.15, (col_y0 + col_y1) / 2, r"$\mathbf{I}=\mathbf{G}^{\!\top}\mathbf{V}$",
       fontsize=10.5, ha="left", va="center", color=ACT)
A.text(cols_x[1] + 1.15, (col_y0 + col_y1) / 2 - 0.55, "(VMM along\nbitlines)",
       fontsize=7.6, ha="left", va="center", color=ACT, style="italic")

# action LIF neurons below the columns
act_y = 3.2
act_lab = [r"$a_1$", r"$a_2$"]
for x, lab in zip(cols_x, act_lab):
    arrow(A, (x, col_y0), (x, act_y + 0.5), color=ACT, lw=1.6, ms=10, z=3)
    A.add_patch(Circle((x, act_y), 0.42, facecolor=OUT, edgecolor=INK, lw=2.0, zorder=4))
    A.text(x, act_y, "LIF", fontsize=8.2, ha="center", va="center", color=INK, fontweight="bold", zorder=6)
    # label beside the neuron (not below) so it clears the WTA arrow on the bitline
    A.text(x + 0.55, act_y, lab, fontsize=9.0, ha="left", va="center", color=ACT, zorder=6)
A.text(cols_x[1] + 1.15, act_y - 0.5, "action\nneurons", fontsize=8.2, ha="left",
       va="center", color=ACT, style="italic")

# winner-take-all action selection
wta_y = 1.55
A.add_patch(FancyBboxPatch((cols_x[0] - 0.9, wta_y - 0.34), 3.3, 0.68,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor="white", edgecolor=INK, lw=1.5, zorder=5))
A.text((cols_x[0] + cols_x[1]) / 2, wta_y, r"WTA: $a=\arg\max_j\!\sum_t s^{\mathrm{post}}_j$",
       fontsize=8.6, ha="center", va="center", color=INK, zorder=6)
for x in cols_x:
    arrow(A, (x, act_y - 0.45), (x, wta_y + 0.36), color=INK, lw=1.3, ms=8, z=3)

# environment / reward (contingent, delayed)
env_x = 9.55
A.add_patch(FancyBboxPatch((env_x - 1.15, wta_y - 0.42), 2.3, 1.5,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor="white", edgecolor=REW, lw=1.8, zorder=5))
A.text(env_x, wta_y + 0.72, "environment", fontsize=8.6, ha="center", va="center", color=REW, zorder=6)
A.text(env_x, wta_y + 0.28, r"$R=\mathbf{1}(a{=}a^\star_s)$", fontsize=8.6, ha="center", va="center", color=INK, zorder=6)
A.text(env_x, wta_y - 0.16, r"delay $D$", fontsize=8.0, ha="center", va="center", color=REW, style="italic", zorder=6)
arrow(A, (cols_x[1] + 1.75, wta_y), (env_x - 1.2, wta_y), color=INK, lw=1.4, ms=10, z=3)

# global third-factor line: R-b broadcast back to every crosspoint
gf_y = 8.95
A.plot([env_x, env_x], [wta_y + 1.08, gf_y], color=REW, lw=1.6, ls=(0, (4, 2)), zorder=2)
A.plot([row_x0 - 0.1, env_x], [gf_y, gf_y], color=REW, lw=1.6, ls=(0, (4, 2)), zorder=2)
A.text((row_x0 + cols_x[1]) / 2 - 0.3, gf_y + 0.26,
       r"global third factor $(R-b)$ broadcast to all synapses",
       fontsize=8.2, ha="center", va="bottom", color=REW)
for x in cols_x:
    arrow(A, (x, gf_y), (x, rows_y[0] + 0.22), color=REW, lw=1.0, ms=7, ls=(0, (2, 2)), z=2)

# (generalization annotation removed -- the caption states the array is S x A)

# ============================ PANEL B: one crosspoint ============================
B.text(-0.2, 8.8, "(b)", fontsize=15, fontweight="bold", va="top")
B.text(5.0, 8.5, "single-layer synapse: one-term update", fontsize=9.0, ha="center", va="top",
       color=INK, fontweight="bold")

# weight device (non-volatile, switching regime) -- differential pair
B.add_patch(FancyBboxPatch((0.7, 5.7), 8.6, 1.9,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor=DEV, edgecolor=INK, lw=1.5))
B.text(5.0, 7.25, "weight device (non-volatile, switching regime)", fontsize=8.2, ha="center", color=INK)
B.text(5.0, 6.45, r"$w_{ij}\;\propto\;G^{+}_{ij}-G^{-}_{ij}$", fontsize=11, ha="center", color=INK)
arrow(B, (5.0, 5.7), (5.0, 5.05), lw=1.5)

# trace device (subthreshold regime) -> local eligibility
B.add_patch(FancyBboxPatch((0.7, 3.0), 8.6, 2.0,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor="white", edgecolor=TRACE, lw=1.6))
B.text(5.0, 4.6, "trace device (subthreshold regime)", fontsize=8.2, ha="center", color=TRACE)
tx = np.linspace(0, 1, 120)
e = (1 - np.exp(-(tx / 0.18) ** 2)) * np.exp(-tx / 0.45); e = e / e.max()
B.plot(1.5 + tx * 3.0, 3.35 + e * 0.85, color=TRACE, lw=1.8)
B.text(5.2, 3.75, r"local eligibility $e_{ij}(t)$", fontsize=8.6, ha="left", va="center", color=TRACE)
B.text(5.2, 3.30, r"retention $\tau_{\mathrm{leak}}$", fontsize=7.8, ha="left", va="center", color=TRACE, style="italic")
arrow(B, (5.0, 3.0), (5.0, 2.35), lw=1.5)

# access transistor performs the reward-gated multiply-write
B.add_patch(FancyBboxPatch((0.4, 0.7), 9.2, 1.6,
            boxstyle="round,pad=0.03,rounding_size=0.1",
            facecolor="white", edgecolor=REW, lw=1.9))
B.text(5.0, 1.92, "reward-gated multiply-write (active element)", fontsize=8.2, ha="center", color=REW)
B.text(5.0, 1.38, r"$\Delta w_{ij}=\eta\,(R-b)\;e_{ij}(t_R)$",
       fontsize=11.5, ha="center", color=INK)
B.text(2.95, 0.88, "global $(R-b)$", fontsize=7.4, ha="center", va="center", color=REW, style="italic")
B.text(6.55, 0.88, "local trace", fontsize=7.4, ha="center", va="center", color=TRACE, style="italic")
# ("no weight transport, no per-synapse gradient" removed -- stated in the text)
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch, Rectangle

# palette matched to crossbar.png
GREEN = "#3aa07a"      # state-input lines
BLUE = "#2f4b8f"       # device synapses / LIF / forward path
LBLUE = "#eaf0fb"      # device-gate fill
RED = "#c0392b"        # global reward broadcast
PURPLE = "#b07cc6"     # fixed-random feedback (DFA), distinct from forward blue
ORANGE = "#e0a93b"     # homeostasis (local activity regulation)
GREY = "#9aa6b2"
INK = "#2b2b2b"

def _box(ax, xy, w, h, fc, ec, **kw):
    ax.add_patch(FancyBboxPatch(xy, w, h, boxstyle="round,pad=0.012,rounding_size=0.02",
                                fc=fc, ec=ec, lw=kw.pop("lw", 1.4), zorder=kw.pop("z", 3), **kw))

def _compound_node(ax, cx, cy, s=0.085):
    """A crossbar synapse drawn as a compact two-tone split square: left half = the
    non-volatile WEIGHT device, right half = the subthreshold TRACE device. Signals the
    two-device compound cell at array scale without three sub-glyphs per tiny node."""
    ax.add_patch(Rectangle((cx - s, cy - s), s, 2 * s, fc="#cbd6e0", ec=INK, lw=0.9, zorder=4))
    ax.add_patch(Rectangle((cx, cy - s), s, 2 * s, fc="white", ec=BLUE, lw=1.0, zorder=4))

def panel_a(ax, tag=True):
    ax.set_xlim(0, 10); ax.set_ylim(-0.3, 7.6); ax.axis("off")
    if tag:
        ax.text(0.1, 7.3, "(a)", fontsize=15, fontweight="bold")

    # ---- global reward broadcast (top, red dashed) -------------------------------
    ax.plot([0.7, 9.4], [7.05, 7.05], color=RED, ls="--", lw=1.6, zorder=2)
    ax.text(5.05, 7.32, r"global third factor $(R-b)$ broadcast to every crosspoint",
            color=RED, fontsize=9.5, ha="center")

    # ---- layer 1 array: inputs (rows) x hidden (cols) ----------------------------
    in_y = [6.15, 5.5]                      # F input lines (2 shown)
    h_x = [2.3, 3.1, 3.9]                   # H hidden columns (3 shown)
    h_lif_y = 4.05
    ax.text(0.15, 5.82, "state\ninputs", color=GREEN, fontsize=9, va="center")
    for x in h_x:                           # hidden bitlines drop to the LIF row
        ax.plot([x, x], [6.45, h_lif_y + 0.2], color=BLUE, lw=1.4, zorder=1)
    for y in in_y:
        ax.add_patch(Circle((1.25, y), 0.11, fc="white", ec=GREEN, lw=1.6, zorder=4))
        ax.plot([1.36, 4.25], [y, y], color=GREEN, lw=2.0, zorder=2)
    for y in in_y:                          # W1 synapses: compound (weight|trace) cells
        for x in h_x:
            _compound_node(ax, x, y, s=0.085)
    ax.text(3.1, 6.62, r"$W_1$ (input$\rightarrow$hidden array)",
            color=BLUE, fontsize=8.5, ha="center")

    # ---- hidden LIF layer (with homeostasis rings) -------------------------------
    for x in h_x:
        ax.add_patch(Circle((x, h_lif_y), 0.32, fc="none", ec=ORANGE, lw=1.3,
                            ls=(0, (2, 1.5)), zorder=4))      # homeostasis ring
        ax.add_patch(Circle((x, h_lif_y), 0.20, fc="#eaf0fb", ec=BLUE, lw=1.5, zorder=5))
        ax.text(x, h_lif_y, "LIF", color=BLUE, fontsize=6.5, ha="center", va="center")
    ax.text(4.35, h_lif_y, "hidden\nneurons", color=BLUE, fontsize=8.5, va="center")
    ax.annotate("", xy=(1.85, h_lif_y + 0.05), xytext=(1.1, h_lif_y + 0.05),
                arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1.5))
    ax.text(0.95, h_lif_y + 0.05, "local\nhomeostasis", color=ORANGE, fontsize=8.0,
            ha="center", va="center")

    # ---- layer 2 array: hidden (rows) x action (cols) ----------------------------
    a_x = [6.4, 7.2]                        # A action columns (2 shown)
    row2_y = [2.45, 2.05, 1.65]             # H hidden rows feeding layer 2
    bus_x = 5.55
    for i, x in enumerate(h_x):             # hidden activity routed to layer-2 rows
        ax.plot([x, bus_x], [h_lif_y - 0.32, h_lif_y - 0.32], color=BLUE, lw=1.0, zorder=1)
    ax.plot([bus_x, bus_x], [h_lif_y - 0.32, row2_y[-1]], color=BLUE, lw=1.0, zorder=1)
    for y in row2_y:
        ax.plot([bus_x, 7.6], [y, y], color=BLUE, lw=1.4, zorder=1)
    for x in a_x:
        ax.plot([x, x], [2.7, 0.55], color=BLUE, lw=1.4, zorder=1)
        for y in row2_y:
            _compound_node(ax, x, y, s=0.080)
    ax.text(6.8, 2.78, r"$W_2$ (hidden$\rightarrow$action array)",
            color=BLUE, fontsize=8.5, ha="center")
    # legend: each array node is a compound cell (weight half + trace half)
    ax.add_patch(Rectangle((8.15, 6.05), 0.14, 0.28, fc="#cbd6e0", ec=INK, lw=0.8, zorder=5))
    ax.add_patch(Rectangle((8.29, 6.05), 0.14, 0.28, fc="white", ec=BLUE, lw=0.9, zorder=5))
    ax.text(8.55, 6.19, "compound cell:\nweight | trace device", color=INK, fontsize=6.6, va="center")

    # ---- action LIF + WTA --------------------------------------------------------
    for x in a_x:
        ax.add_patch(Circle((x, 0.35), 0.20, fc="#eaf0fb", ec=BLUE, lw=1.5, zorder=5))
        ax.text(x, 0.35, "LIF", color=BLUE, fontsize=6.5, ha="center", va="center")
    ax.text(7.65, 0.35, "action\nneurons", color=BLUE, fontsize=8.5, va="center")

    # ---- DFA fixed-random feedback array (purple) --------------------------------
    # output error projected to the hidden layer through a FIXED RANDOM matrix (no W2^T)
    fb = FancyArrowPatch((6.55, 0.15), (3.1, 3.55), connectionstyle="arc3,rad=0.30",
                         arrowstyle="-|>", mutation_scale=13, color=PURPLE, lw=1.9,
                         ls=(0, (5, 2)), zorder=2)
    ax.add_patch(fb)
    ax.text(3.05, 1.15, r"fixed random feedback $B$:  $L_h = B\,L_a$" "\n"
            r"per-neuron credit, no $W_2^{\top}$ transport",
            color=PURPLE, fontsize=8.0, ha="center", va="center")

    # (array-dimension annotation removed -- the caption describes the two
    #  stacked W_1, W_2 arrays flanking the hidden layer)

def panel_b(ax, tag=True):
    ax.set_xlim(0, 10); ax.set_ylim(0, 7.4); ax.axis("off")
    if tag:
        ax.text(0.1, 7.05, "(b)", fontsize=15, fontweight="bold")
    ax.text(5.0, 6.95, "hidden-layer synapse: two-term update", fontsize=10.5, fontweight="bold", ha="center")

    # weight device (non-volatile)
    _box(ax, (1.4, 5.45), 7.2, 0.95, "#f1f3f5", "#444")
    ax.text(5.0, 6.12, r"weight device (non-volatile, switching)", fontsize=9.0, ha="center")
    ax.text(5.0, 5.72, r"$w_{ij}\;\propto\;G^{+}_{ij}-G^{-}_{ij}$", fontsize=11, ha="center")
    # arrows flow top->bottom (weight -> gate -> update): arrowhead (xy) at the LOWER box,
    # tail (xytext) at the upper box, so the head points DOWN into the next stage.
    ax.annotate("", xy=(5.0, 4.95), xytext=(5.0, 5.45),
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.4))

    # trace device (subthreshold) / eligibility
    _box(ax, (1.4, 3.85), 7.2, 1.05, LBLUE, BLUE)
    ax.text(5.0, 4.62, "trace device (subthreshold regime)", color=BLUE, fontsize=9.0, ha="center")
    ax.text(5.0, 4.18, r"local eligibility $e_{ij}(t)$,  retention $\tau_{\mathrm{leak}}$",
            color=BLUE, fontsize=8.5, ha="center", style="italic")
    # tiny eligibility glyph
    tt = np.linspace(0, 1, 60)
    gl = (1 - np.exp(-(tt / 0.18) ** 2)) * np.exp(-tt / 0.5)
    ax.plot(1.7 + tt * 0.9, 4.0 + gl * 0.55, color=BLUE, lw=1.3)
    ax.annotate("", xy=(5.0, 3.4), xytext=(5.0, 3.85),
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.4))

    # local update box: two contributions; reward-gated multiply-write by an active
    # element (per-cell selector/transistor or shared peripheral CMOS), not the memristors
    _box(ax, (1.0, 0.5), 8.0, 2.75, "white", RED, lw=1.6)
    ax.text(5.0, 2.92, r"reward-gated update $\Delta w_{ij}$ (active element)", color=RED, fontsize=9.0, ha="center")
    # term 1: feedback-aligned credit
    ax.text(5.0, 2.42, r"$\eta\,(R-b)\,[B\,L]_{j}\;\,e_{ij}(t_R)$",
            fontsize=11, ha="center", color=PURPLE)
    ax.text(5.0, 2.08, "per-neuron credit (DFA)", fontsize=7.6, ha="center",
            color=PURPLE, style="italic")
    # plus
    ax.text(5.0, 1.62, r"$+$", fontsize=12, ha="center", color=INK)
    # term 2: homeostatic regulation
    ax.text(5.0, 1.18, r"$\eta_h\,(\rho^{\ast}-\bar\rho_j)\;\,w_{ij}$",
            fontsize=11, ha="center", color=ORANGE)
    ax.text(5.0, 0.84, "local homeostasis", fontsize=7.6, ha="center",
            color=ORANGE, style="italic")
    # (grey "rho_j: neuron's own activity / rho*: shared target" gloss removed
    #  -- both symbols are defined in the surrounding text, Eq. 7.14 paragraph)

    # local/global annotations
    ax.text(2.0, 0.12, "global scalar", color=RED, fontsize=7.4, ha="center")
    ax.text(5.0, 0.12, "fixed-random, local", color=PURPLE, fontsize=7.4, ha="center")
    ax.text(8.0, 0.12, "self-activity, local", color=ORANGE, fontsize=7.4, ha="center")

# combined two-panel version (kept for reference)
fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.0, 4.0),
                               gridspec_kw={"width_ratios": [1.32, 1.0]})
panel_a(axA)
panel_b(axB)
fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.04, wspace=0.04)

plt.show()

# split panels: each rendered standalone at full width so the schematic text is
# legible when the figure is placed at ~0.7-0.8\textwidth in the manuscript.
figA = plt.figure(figsize=(7.2, 5.2))
axA1 = figA.add_axes([0.01, 0.02, 0.98, 0.96])
panel_a(axA1, tag=False)
outA = os.path.join(OUT, "fig_deep_crossbar_a.png")
plt.show()
figB = plt.figure(figsize=(6.2, 4.6))
axB1 = figB.add_axes([0.01, 0.02, 0.98, 0.96])
panel_b(axB1, tag=False)
outB = os.path.join(OUT, "fig_deep_crossbar_b.png")
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle, FancyBboxPatch, Patch

INK    = "#222222"
POS    = "#c0392b"     # positive feedback / runaway (red)
HOMEO  = "#e0a93b"     # homeostasis (orange)
EEG    = "#b07cc6"     # EEG / noisy reward (indigo)
NET    = "#2f4b8f"
GREEN  = "#3aa07a"
NEU    = "#9aa6b2"
SET    = "#3aa07a"     # set-point

def feedback_ring(ax, cx, cy, r, color, sign, ccw=True):
    """A circular feedback arrow with a +/- sign in the middle."""
    th0, th1 = (200, -110) if ccw else (-20, 250)
    arc = np.linspace(np.deg2rad(th0), np.deg2rad(th1), 100)
    ax.plot(cx+r*np.cos(arc), cy+r*np.sin(arc), color=color, lw=2.6, zorder=4)
    # arrowhead at the end
    a = arc[-1]; da = (arc[-1]-arc[-2])
    ax.add_patch(FancyArrowPatch((cx+r*np.cos(a-da), cy+r*np.sin(a-da)),
                 (cx+r*np.cos(a), cy+r*np.sin(a)), arrowstyle="-|>",
                 mutation_scale=16, color=color, lw=2.6, zorder=5))
    ax.text(cx, cy, sign, ha="center", va="center", fontsize=20, fontweight="bold",
            color=color, zorder=6)

def panel_a(ax):
    ax.text(-0.04, 1.08, "(a)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Associative plasticity is positive feedback", fontsize=9.4, fontweight="bold", pad=6)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
    # the loop: fires more -> weights grow -> fires more
    feedback_ring(ax, 5.0, 6.3, 1.7, POS, "$+$", ccw=True)
    ax.text(5.0, 8.5, "neuron fires more", ha="center", fontsize=8.0, color=POS)
    ax.text(8.0, 6.3, "its weights\ngrow", ha="center", fontsize=8.0, color=POS)
    ax.text(2.0, 6.3, "drive\nrises", ha="center", fontsize=8.0, color=POS)
    # runaway trace
    t = np.linspace(0, 1, 100)
    run = 1/(1+np.exp(-(t-0.45)*12))           # logistic blow-up to the rail
    ax.plot(1.2+t*7.6, 0.8+run*2.6, color=POS, lw=2.2)
    ax.axhline(0.8+2.6, xmin=0.12, xmax=0.88, color=NEU, lw=0.8, ls=(0,(3,2)))
    ax.text(8.9, 0.8+2.6, "rail", fontsize=6.8, color=NEU, va="center")
    ax.text(5.0, 0.35, "left alone: a hidden unit locks high $\\to$ policy collapses",
            ha="center", fontsize=7.4, color=POS, style="italic")

def panel_b(ax):
    ax.text(-0.04, 1.08, "(b)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Homeostasis is the negative-feedback counterweight", fontsize=9.4, fontweight="bold", pad=6)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
    feedback_ring(ax, 5.0, 6.3, 1.7, HOMEO, "$-$", ccw=False)
    ax.text(5.0, 8.5, "rate above $\\rho^{\\!*}$", ha="center", fontsize=8.0, color=HOMEO)
    ax.text(8.1, 6.3, "weights\npulled down", ha="center", fontsize=8.0, color=HOMEO)
    ax.text(1.9, 6.3, "rate\nreturns", ha="center", fontsize=8.0, color=HOMEO)
    # rate settling to set-point
    t = np.linspace(0, 1, 200)
    settle = 0.5 + 0.5*np.exp(-t*5)*np.cos(t*14)   # damped to set-point
    ax.plot(1.2+t*7.6, 0.8+settle*2.6, color=HOMEO, lw=2.2)
    ax.axhline(0.8+0.5*2.6, xmin=0.12, xmax=0.88, color=SET, lw=1.0, ls=(0,(4,2)))
    ax.text(8.9, 0.8+0.5*2.6, r"$\rho^{\!*}$", fontsize=8.5, color=SET, va="center")
    ax.text(5.0, 0.35, "each neuron holds its own rate near a shared set-point",
            ha="center", fontsize=7.4, color=HOMEO, style="italic")

def panel_c(ax):
    rng = np.random.RandomState(3)
    n = 200; trial = np.arange(n)
    # noisy reward strip at the top
    rew = 0.5 + 0.5*np.sign(np.sin(trial*0.5)) * (rng.rand(n) < 0.85)  # noisy +/- reward
    ax.plot(trial, 9.3 + 0.001*trial, color="none")
    for i in range(0, n, 2):
        c = EEG if rew[i] > 0.5 else NEU
        ax.plot([trial[i], trial[i]], [9.0, 9.0+0.35*(0.4+0.6*(rew[i]>0.5))], color=c, lw=0.8, alpha=0.7)
    ax.text(n/2, 9.7, "noisy single-trial reward (EEG-decoded)", ha="center", fontsize=8.0, color=EEG)

    # without homeostasis: firing rate diverges to a rail (positive feedback unchecked)
    rho = 0.5; x = rho; traj_no = []
    for i in range(n):
        x += 0.018*(rew[i]-0.5)*4 + 0.03*(x-rho) + 0.01*rng.randn()  # pos feedback (x-rho) destabilises
        x = np.clip(x, 0, 1); traj_no.append(x)
    ax.plot(trial, np.array(traj_no)*6.8+0.8, color=POS, lw=2.2,
            label="no homeostasis (positive feedback runs away)")

    # with homeostasis: bounded near set-point (negative-feedback counterweight)
    x = rho; traj_h = []
    for i in range(n):
        x += 0.018*(rew[i]-0.5)*4 + 0.03*(x-rho) - 0.16*(x-rho) + 0.01*rng.randn()  # neg feedback dominates
        x = np.clip(x, 0, 1); traj_h.append(x)
    ax.plot(trial, np.array(traj_h)*6.8+0.8, color=HOMEO, lw=2.4,
            label="with homeostasis (held near the set-point)")

    ax.axhline(rho*6.8+0.8, color=SET, lw=1.0, ls=(0,(4,2)))
    ax.text(n+2, rho*6.8+0.8, r"set-point $\rho^{\!*}$", fontsize=8.5, color=SET, va="center")
    ax.text(n+2, 1.0*6.8+0.8-0.18, "saturation rail", fontsize=7.4, color=NEU, va="center")
    # name the two control regimes directly on the curves
    ax.annotate("runaway: unit locks high,\npolicy collapses", xy=(150, traj_no[150]*6.8+0.8),
                xytext=(96, 8.1), fontsize=7.6, color=POS, va="center",
                arrowprops=dict(arrowstyle="->", color=POS, lw=1.0))
    ax.annotate("bounded: extra reward noise\nabsorbed", xy=(150, traj_h[150]*6.8+0.8),
                xytext=(70, 2.0), fontsize=7.6, color=HOMEO, va="center",
                arrowprops=dict(arrowstyle="->", color=HOMEO, lw=1.0))

    ax.set_xlim(0, n+30); ax.set_ylim(0, 10.2)
    ax.set_xlabel("trial", fontsize=9.5)
    ax.set_ylabel("hidden-unit firing rate", fontsize=9.5)
    ax.set_yticks([]); ax.tick_params(labelsize=8.5)
    ax.legend(loc="lower left", fontsize=8.2, framealpha=0.95, bbox_to_anchor=(0.01, 0.02))

fig, ax = plt.subplots(figsize=(7.2, 4.4))
panel_c(ax)
fig.suptitle("Why homeostasis stabilises learning under a noisy reward",
             fontsize=11.5, fontweight="bold", y=0.98)
fig.subplots_adjust(left=0.07, right=0.99, top=0.90, bottom=0.12)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ---- palette (matches the manuscript green-blue scheme) ---------------------
TEAL = "#3aa07a"   # device measured bands / markers
INDIGO = "#2f4b8f"  # tuned RL retention settings
BRICK = "#c0392b"  # biological window accent
INK = "#2b2b2b"
GRID = "#c8d0d8"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.edgecolor": INK,
    "text.color": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})

# ---- data (all real; see module docstring) ----------------------------------
BIO = (0.3, 10.0)           # biological eligibility window (s)
TAU_R = (1.9, 14.5)         # MEASURED cascade rise constant (s)
TAU_D = (36.0, 537.0)       # MEASURED space-charge decay constant (s)
TAU_LEAK_MEAS = 1.3         # MEASURED held-bias trap-discharge retention (ITO median, s)
TAU_LEAK_FF = 3.6           # field-free retention (Poole-Frenkel extrapolation, s)
TAU_LEAK = [5, 10, 20]      # MODEL-SET retention settings swept in RL (s)

# the genuine overlap stripe = biological window  AND  device measured dynamics
# i.e. [max(0.3, 1.3), min(10, 14.5)] = [1.3, 10]
OVERLAP = (max(BIO[0], TAU_LEAK_MEAS), min(BIO[1], TAU_R[1]))

# y-rows (top to bottom)
Y_BIO = 3
Y_R = 2
Y_D = 1
Y_LEAK = 0
BAR_H = 0.30  # half-height of bands

fig, ax = plt.subplots(figsize=(7.2, 3.0))

# ---- key-message overlap stripe (the true intersection ~1.3-10 s) -----------
ax.axvspan(OVERLAP[0], OVERLAP[1], color=TEAL, alpha=0.08, zorder=0)
ax.text(np.sqrt(OVERLAP[0] * OVERLAP[1]), 4.05,
        "device dynamics meet\nthe biological window",
        ha="center", va="center", fontsize=7.0, style="italic", color="#5a6b62",
        linespacing=1.05, zorder=5)

# ---- row 1: biological eligibility window (hatched band) --------------------
ax.fill_between(BIO, Y_BIO - BAR_H, Y_BIO + BAR_H, facecolor="none",
                edgecolor=BRICK, hatch="////", linewidth=1.1, zorder=3)
ax.text(BIO[1] * 1.35, Y_BIO,
        "biological eligibility window (~0.3-10 s)",
        va="center", ha="left", fontsize=8.5, color=BRICK)

# ---- row 2: MEASURED device rise tau_r (solid band) -------------------------
ax.fill_between(TAU_R, Y_R - BAR_H, Y_R + BAR_H, color=TEAL, alpha=0.85,
                zorder=3)
ax.text(TAU_R[1] * 1.35, Y_R,
        r"device rise  $\tau_r$  (measured, 1.9-14.5 s)",
        va="center", ha="left", fontsize=8.5, color=INK)

# ---- row 3: MEASURED device decay tau_d (solid band, lighter shade) ---------
ax.fill_between(TAU_D, Y_D - BAR_H, Y_D + BAR_H, color=TEAL, alpha=0.45,
                zorder=3)
ax.text(TAU_D[0] * 0.62, Y_D,
        r"device decay  $\tau_d$  (measured, 36-537 s)",
        va="center", ha="right", fontsize=8.5, color=INK)

# ---- row 4: tau_leak -- MEASURED retention (filled) + RL sweep (open) --------
# the held-bias ITO retention (filled) and its field-free Poole-Frenkel
# extrapolation (filled, lighter): the credit window the paper rests on, both
# sitting inside the biological window. An arrow marks the field correction.
ax.annotate("", xy=(TAU_LEAK_FF, Y_LEAK), xytext=(TAU_LEAK_MEAS, Y_LEAK),
            arrowprops=dict(arrowstyle="-|>", color="#5a6b62", lw=1.1,
                            shrinkA=5, shrinkB=5), zorder=4)
ax.plot([TAU_LEAK_MEAS], [Y_LEAK], marker="o", ms=9.5, mfc=TEAL,
        mec=INK, mew=1.0, zorder=5)
ax.annotate(r"$1.3$ (held bias)", (TAU_LEAK_MEAS, Y_LEAK),
            textcoords="offset points", xytext=(0, 10), ha="center",
            fontsize=6.8, color=INK)
ax.plot([TAU_LEAK_FF], [Y_LEAK], marker="o", ms=9.5, mfc=TEAL, alpha=0.6,
        mec=INK, mew=1.0, zorder=5)
ax.annotate(r"$3.6$ (field-free)", (TAU_LEAK_FF, Y_LEAK),
            textcoords="offset points", xytext=(0, -14), ha="center",
            fontsize=6.8, color=INK)
# the tunable retention settings swept in the RL experiments (open markers)
for t in TAU_LEAK:
    ax.plot([t], [Y_LEAK], marker="o", ms=8.5, mfc="white",
            mec=INDIGO, mew=1.6, zorder=4)
    ax.annotate(f"{t}", (t, Y_LEAK), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=7, color=INDIGO)
ax.text(TAU_LEAK[-1] * 1.6, Y_LEAK,
        r"retention  $\tau_{\mathrm{leak}}$  (measured $+$ tuned)",
        va="center", ha="left", fontsize=8.5, color=INDIGO)

# ---- axes cosmetics ---------------------------------------------------------
ax.set_xscale("log")
ax.set_xlim(0.1, 1000)
ax.set_ylim(-0.7, 4.5)
ax.set_xlabel("time constant / window (s)")
ax.set_yticks([])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="x", which="both", length=3)
ax.grid(True, axis="x", which="major", color=GRID, lw=0.6, zorder=0)
ax.set_axisbelow(True)

# ---- legend: MEASURED vs MODEL-SET vs BIOLOGICAL ----------------------------
legend_handles = [
    Patch(facecolor=TEAL, alpha=0.85, label="measured device dynamics"),
    Line2D([0], [0], marker="o", color="none", mfc=TEAL, mec=INK,
           mew=1.0, ms=8, label=r"measured $\tau_{\mathrm{leak}}$ (credit window)"),
    Line2D([0], [0], marker="o", color="none", mfc="white", mec=INDIGO,
           mew=1.6, ms=8, label="tuned retention (RL)"),
    Patch(facecolor="none", edgecolor=BRICK, hatch="////",
          label="biological (literature)"),
]
ax.legend(handles=legend_handles, loc="upper right",
          bbox_to_anchor=(1.0, 0.99), frameon=False, fontsize=7.5,
          handlelength=1.6, borderaxespad=0.2, labelspacing=0.35)

fig.tight_layout()

plt.show()
plt.close(fig)

In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ---- palette (matches the manuscript green-indigo scheme) -----------------------
TEAL = "#3aa07a"    # device measured bands / markers
INDIGO = "#2f4b8f"  # tuned RL retention settings
BRICK = "#c0392b"   # biological window accent
INK = "#2b2b2b"
GRID = "#c8d0d8"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.edgecolor": INK,
    "text.color": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})

# ---- data (all real; per the the biological-grounding section caption) -----------------------------
BIO = (0.3, 10.0)           # biological eligibility window (s)
TAU_R = (1.9, 14.5)         # MEASURED cascade rise constant (s)
TAU_D = (36.0, 537.0)       # MEASURED space-charge decay constant (s)
TAU_LEAK_MEAS = 1.3         # MEASURED held-bias trap-discharge retention (ITO median, s)
TAU_LEAK_FF = 3.6           # field-free retention (Poole-Frenkel extrapolation, s)
TAU_LEAK = [5, 10, 20]      # MODEL-SET retention settings swept in RL (s)

OVERLAP = (max(BIO[0], TAU_LEAK_MEAS), min(BIO[1], TAU_R[1]))  # true intersection ~1.3-10 s

Y_BIO = 3
Y_R = 2
Y_D = 1
Y_LEAK = 0
BAR_H = 0.30

fig, ax = plt.subplots(figsize=(7.2, 3.0))

# ---- key-message overlap stripe (the true intersection ~1.3-10 s), no text ----
ax.axvspan(OVERLAP[0], OVERLAP[1], color=TEAL, alpha=0.08, zorder=0)

# ---- row 1: biological eligibility window (hatched band) --------------------
ax.fill_between(BIO, Y_BIO - BAR_H, Y_BIO + BAR_H, facecolor="none",
                edgecolor=BRICK, hatch="////", linewidth=1.1, zorder=3)

# ---- row 2: MEASURED device rise tau_r (solid band) -------------------------
ax.fill_between(TAU_R, Y_R - BAR_H, Y_R + BAR_H, color=TEAL, alpha=0.85, zorder=3)

# ---- row 3: MEASURED device decay tau_d (solid band, lighter shade) ---------
ax.fill_between(TAU_D, Y_D - BAR_H, Y_D + BAR_H, color=TEAL, alpha=0.45, zorder=3)

# ---- row 4: tau_leak -- MEASURED retention (filled) + RL sweep (open) --------
# held-bias ITO retention (filled) and its field-free Poole-Frenkel extrapolation
# (filled, lighter); an arrow marks the field correction. No inline numbers.
ax.annotate("", xy=(TAU_LEAK_FF, Y_LEAK), xytext=(TAU_LEAK_MEAS, Y_LEAK),
            arrowprops=dict(arrowstyle="-|>", color="#5a6b62", lw=1.1,
                            shrinkA=5, shrinkB=5), zorder=4)
ax.plot([TAU_LEAK_MEAS], [Y_LEAK], marker="o", ms=9.5, mfc=TEAL,
        mec=INK, mew=1.0, zorder=5)
ax.plot([TAU_LEAK_FF], [Y_LEAK], marker="o", ms=9.5, mfc=TEAL, alpha=0.6,
        mec=INK, mew=1.0, zorder=5)
# the tunable retention settings swept in the RL experiments (open markers), no numbers
for t in TAU_LEAK:
    ax.plot([t], [Y_LEAK], marker="o", ms=8.5, mfc="white",
            mec=INDIGO, mew=1.6, zorder=4)

# ---- axes cosmetics ---------------------------------------------------------
ax.set_xscale("log")
ax.set_xlim(0.1, 1000)
ax.set_ylim(-0.7, 4.5)
ax.set_xlabel("time constant / window (s)")
ax.set_yticks([])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="x", which="both", length=3)
ax.grid(True, axis="x", which="major", color=GRID, lw=0.6, zorder=0)
ax.set_axisbelow(True)

# ---- legend: everything the removed inline labels used to carry --------------
legend_handles = [
    Patch(facecolor="none", edgecolor=BRICK, hatch="////",
          label=r"biological eligibility window ($\sim$0.3--10 s)"),
    Patch(facecolor=TEAL, alpha=0.85, label=r"device rise $\tau_r$ (measured)"),
    Patch(facecolor=TEAL, alpha=0.45, label=r"device decay $\tau_d$ (measured)"),
    Line2D([0], [0], marker="o", color="none", mfc=TEAL, mec=INK,
           mew=1.0, ms=8, label=r"$\tau_{\mathrm{leak}}$, held-bias (measured)"),
    Line2D([0], [0], marker="o", color="none", mfc=TEAL, alpha=0.6, mec=INK,
           mew=1.0, ms=8, label=r"$\tau_{\mathrm{leak}}$, field-free (extrapolated)"),
    Line2D([0], [0], marker="o", color="none", mfc="white", mec=INDIGO,
           mew=1.6, ms=8, label=r"$\tau_{\mathrm{leak}}$, tuned (RL settings)"),
]
ax.legend(handles=legend_handles, loc="upper right",
          bbox_to_anchor=(1.0, 1.02), frameon=False, fontsize=7.2,
          handlelength=1.6, borderaxespad=0.2, labelspacing=0.4)

fig.tight_layout()
plt.show()
plt.close(fig)